# Mount Drive and Import Data

In [ ]:
from google.colab import drive
import pandas as pd
import os
import warnings

warnings.filterwarnings("ignore")

drive.mount("/content/drive")

BASE_PATH = f"/content/drive/MyDrive/COMP721 Project"

DATA_PATH = f"{BASE_PATH}/data"

print("Paths and libraries ready.")

file_path = f"{DATA_PATH}/cardio_train.csv"

if os.path.exists(file_path):
    print("\nDataset found! Loading data...")
    df = pd.read_csv(file_path, sep=";")

    df["age_years"] = df["age"] / 365.25
    print("Data loaded successfully.")

    display(df.head())

else:
    print(f"\nERROR: Dataset not found at {file_path}")
    print(
        "Check that 'cardio_train.csv' is inside the project's 'data' folder in My Drive."
    )
    raise FileNotFoundError("Dataset path is incorrect.")

# Data Preprocessing

In [ ]:
import os
import warnings
from google.colab import drive
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

drive.mount("/content/drive")

BASE_PATH = "/content/drive/MyDrive/COMP721 Project"
DATA_PATH = f"{BASE_PATH}/data"
file_path = f"{DATA_PATH}/cardio_train.csv"

if os.path.exists(file_path):
    df = pd.read_csv(file_path, sep=";")

    # Convert age from days to years
    df["age_years"] = df["age"] / 365.25

    print(f"Raw dataset loaded: {df.shape[0]:,} rows, {df.shape[1]} columns.")
else:
    raise FileNotFoundError(f"Dataset not found at {file_path}.")

# Duplicate removal
print("\nStarting Data Preprocessing...\n")
print(f"Original dataset shape: {df.shape}")

# Exclude unique identifier and derived age_years column
cols_for_dupe_check = [c for c in df.columns if c not in ("id", "age_years")]

duplicates_count = df.duplicated(subset=cols_for_dupe_check, keep="first").sum()

df_clean = df.drop_duplicates(subset=cols_for_dupe_check, keep="first").copy()

print(
    f"Removed {duplicates_count} duplicated clinical records "
    f"({duplicates_count / len(df) * 100:.2f}% of the original data)."
)

# Missing value analysis
missing_by_column = df_clean.isnull().sum()
missing_count = int(missing_by_column.sum())

if missing_count == 0:
    print("Missing values: 0 found. No imputation required.")
else:
    print(f"Missing values found: {missing_count}.")
    print(missing_by_column[missing_by_column > 0])

    df_clean = df_clean.dropna().copy()

# Inconsistent values and outlier treatment
rows_before = len(df_clean)

print(f"\nRows entering outlier/inconsistency treatment: {rows_before:,}")

# Correct negative blood-pressure signs
neg_bp_mask = (df_clean["ap_hi"] < 0) | (df_clean["ap_lo"] < 0)

neg_bp_count = int(neg_bp_mask.sum())

df_clean["ap_hi"] = df_clean["ap_hi"].abs()
df_clean["ap_lo"] = df_clean["ap_lo"].abs()

print(
    f"a) Corrected {neg_bp_count} rows containing negative blood-pressure "
    f"values using a sign-error assumption."
)

# Remove internally inconsistent blood-pressure records
before = len(df_clean)

df_clean = df_clean[df_clean["ap_hi"] > df_clean["ap_lo"]].copy()

print(f"b) Dropped {before - len(df_clean)} rows where ap_lo >= ap_hi.")

# Apply broad physiological plausibility thresholds
clinical_ranges = {
    "height": (125, 215),
    "weight": (40, 200),
    "ap_hi": (70, 250),
    "ap_lo": (40, 150),
}

for label, col in [
    ("c1", "height"),
    ("c2", "weight"),
    ("c3", "ap_hi"),
    ("c4", "ap_lo"),
]:
    lo, hi = clinical_ranges[col]

    before = len(df_clean)

    df_clean = df_clean[df_clean[col].between(lo, hi, inclusive="both")].copy()

    removed = before - len(df_clean)

    print(f"{label}) Dropped {removed} rows with {col} outside [{lo}, {hi}].")


rows_after = len(df_clean)

print(
    f"\nTotal rows removed during inconsistency/outlier treatment: "
    f"{rows_before - rows_after} "
    f"({(rows_before - rows_after) / rows_before * 100:.2f}%)"
)

print(f"Rows remaining: {rows_after:,}")

# Catergorical encoding

# Raw gender coding:
#   1 = Women
#   2 = Men
# Remap to:
#   0 = Women
#   1 = Men

df_clean["gender"] = df_clean["gender"].map({1: 0, 2: 1})

if df_clean["gender"].isnull().any():
    raise ValueError("Unexpected value detected in 'gender' during encoding.")

print("\nCategorical encoding complete: 'gender' mapped to 0 (Women) / 1 (Men).")

print(
    "'cholesterol' and 'gluc' retained as ordinal integer codes; "
    "'smoke', 'alco', and 'active' are already binary."
)

# Remove non-predictive columns
# id is an identifier and should not be used as a predictor.
# age is replaced by age_years.
df_clean = df_clean.drop(columns=["id", "age"], errors="ignore")

# Final validation
df_clean = df_clean.reset_index(drop=True)

final_duplicate_count = df_clean.duplicated().sum()

print(f"\nExact duplicate records after final preprocessing: {final_duplicate_count}")

if final_duplicate_count > 0:
    print(
        "Warning: exact duplicates remain after preprocessing. "
        "Inspect them before deciding whether they should be removed."
    )

print(f"\nFinal cleaned dataset shape: {df_clean.shape}")

print(
    f"Rows removed overall: "
    f"{len(df):,} -> {len(df_clean):,} "
    f"({(len(df) - len(df_clean)) / len(df) * 100:.2f}% "
    f"of the original dataset)"
)

print("\nClass balance after cleaning:")
print(df_clean["cardio"].value_counts(normalize=True).sort_index().mul(100).round(2))

display(df_clean.describe().T[["count", "mean", "std", "min", "max"]])

output_path = f"{DATA_PATH}/cardio_train_cleaned.csv"

df_clean.to_csv(output_path, index=False)

print(f"\nCleaned dataset saved to:\n{output_path}")

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive

sns.set_theme(style="whitegrid", palette="muted")

drive.mount("/content/drive")
BASE_PATH = "/content/drive/MyDrive/COMP721 Project"
DATA_PATH = f"{BASE_PATH}/data"
file_path = f"{DATA_PATH}/cardio_train.csv"  # Original dataset

if os.path.exists(file_path):
    df_raw = pd.read_csv(file_path, sep=";")
    print(f"Original raw dataset loaded successfully: {df_raw.shape[0]:,} rows.")

    # Generate Boxplots
    plt.figure(figsize=(16, 8))
    for i, feature in enumerate(["height", "weight", "ap_hi", "ap_lo"]):
        plt.subplot(1, 4, i + 1)
        sns.boxplot(y=df_raw[feature], color="lightblue")
        plt.title(f"Boxplot of {feature}")

    plt.suptitle(
        "Raw Data Outliers (Original Pre-Cleaned Dataset)", y=1.05, fontweight="bold"
    )
    plt.tight_layout()
    plt.show()
else:
    print(f"File not found: {file_path}. Please check your Drive path.")

# Exploratory Data Analysis

In [ ]:
import os
import warnings

from google.colab import drive

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="muted")

drive.mount("/content/drive")

BASE_PATH = "/content/drive/MyDrive/COMP721 Project"

DATA_PATH = f"{BASE_PATH}/data"
file_path = f"{DATA_PATH}/cardio_train_cleaned.csv"

if os.path.exists(file_path):
    df = pd.read_csv(file_path)

    print("Cleaned data loaded successfully.")
    display(df.head())

else:
    raise FileNotFoundError(
        f"Dataset not found at {file_path}. "
        f"Check BASE_PATH and the project folder in My Drive."
    )

print("\n" + "=" * 70)
print("DATASET SHAPE AND STRUCTURE")
print("=" * 70)

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

print("\nData types:")
print(df.dtypes)

print("\n" + "=" * 70)
print("MISSING VALUES")
print("=" * 70)

missing = df.isnull().sum()

if missing.sum() > 0:
    missing_pct = (missing / len(df) * 100).round(2)

    missing_summary = pd.DataFrame(
        {"Missing Count": missing, "Missing Percentage (%)": missing_pct}
    )

    display(missing_summary[missing_summary["Missing Count"] > 0])

else:
    print("No missing values found.")

print("\n" + "=" * 70)
print("DUPLICATE AND CONFLICTING RECORDS")
print("=" * 70)

exact_duplicate_count = int(df.duplicated().sum())

print(f"Exact duplicate clinical records: {exact_duplicate_count}")

feature_cols = [col for col in df.columns if col != "cardio"]

conflicting_rows = df.groupby(feature_cols, dropna=False, group_keys=False).filter(
    lambda group: group["cardio"].nunique() > 1
)

num_conflicting_rows = len(conflicting_rows)

num_conflicting_profiles = len(conflicting_rows.drop_duplicates(subset=feature_cols))

print(
    f"Conflicting records "
    f"(identical predictors but different CVD labels): "
    f"{num_conflicting_rows}"
)

print(
    f"Unique clinical profiles with conflicting CVD labels: {num_conflicting_profiles}"
)

if num_conflicting_rows > 0:
    print("\nClass distribution among conflicting records:")

    print(conflicting_rows["cardio"].value_counts().sort_index())

# TARGET VARIABLE CLASS DISTRIBUTION

print("\n" + "=" * 70)
print("TARGET VARIABLE (cardio) CLASS DISTRIBUTION")
print("=" * 70)

class_counts = df["cardio"].value_counts().sort_index()

class_pct = df["cardio"].value_counts(normalize=True).sort_index().mul(100).round(2)

class_summary = pd.DataFrame({"Count": class_counts, "Percentage (%)": class_pct})

class_summary.index = class_summary.index.map({0: "No CVD (0)", 1: "Has CVD (1)"})

display(class_summary)

class_balance_ratio = class_counts.loc[0] / class_counts.loc[1]

print(f"\nClass balance ratio: {class_balance_ratio:.2f}:1")

print(
    "The target remains effectively balanced; "
    "therefore, imbalance-specific resampling is not required."
)

print("\n" + "=" * 70)
print("DESCRIPTIVE STATISTICS")
print("=" * 70)


continuous_cols = ["age_years", "height", "weight", "ap_hi", "ap_lo"]

categorical_cols = [
    "gender",
    "cholesterol",
    "gluc",
    "smoke",
    "alco",
    "active",
    "cardio",
]

# Continuous Features

print("\n--- Continuous Features ---")

continuous_summary = (
    df[continuous_cols]
    .describe()
    .T[["count", "mean", "std", "min", "25%", "50%", "75%", "max"]]
)

display(continuous_summary.round(2))

# Ordinal and Binary Features

print("\n--- Ordinal and Binary Features ---")

for col in categorical_cols:
    counts = df[col].value_counts().sort_index()

    percentages = df[col].value_counts(normalize=True).sort_index().mul(100).round(2)

    summary_df = pd.DataFrame({"Count": counts, "Percentage (%)": percentages})

    print(f"\nFeature: {col}")
    display(summary_df)

viz_df = df.copy()


viz_df["gender"] = viz_df["gender"].map({0: "Women", 1: "Men"})


viz_df["cholesterol"] = viz_df["cholesterol"].map(
    {1: "Normal", 2: "Above Normal", 3: "Well Above"}
)


viz_df["gluc"] = viz_df["gluc"].map({1: "Normal", 2: "Above Normal", 3: "Well Above"})


binary_map = {0: "No", 1: "Yes"}


viz_df["smoke"] = viz_df["smoke"].map(binary_map)

viz_df["alco"] = viz_df["alco"].map(binary_map)

viz_df["active"] = viz_df["active"].map(binary_map)


viz_df["cardio_label"] = viz_df["cardio"].map({0: "No CVD (0)", 1: "Has CVD (1)"})

# TARGET DISTRIBUTION

plt.figure(figsize=(6, 4))

ax = sns.countplot(data=viz_df, x="cardio_label", palette="viridis")

ax.margins(y=0.18)

plt.title("Class Distribution of Target Variable (Cardio)", pad=15)

plt.xlabel("Cardiovascular Disease Diagnosis")

plt.ylabel("Count")

total = len(viz_df)

for patch in ax.patches:
    count = patch.get_height()

    percentage = 100 * count / total

    ax.annotate(
        f"{count:,.0f}\n({percentage:.2f}%)",
        (patch.get_x() + patch.get_width() / 2, count),
        ha="center",
        va="bottom",
    )

plt.tight_layout()
plt.show()

# CATEGORICAL FEATURE DISTRIBUTIONS

categorical_features = ["gender", "cholesterol", "gluc", "smoke", "alco", "active"]


fig, axes = plt.subplots(2, 3, figsize=(16, 9))


for ax, feature in zip(axes.flatten(), categorical_features):
    sns.countplot(data=viz_df, x=feature, palette="muted", ax=ax)

    ax.set_title(f"Distribution of {feature.capitalize()}")

    ax.set_xlabel("")
    ax.set_ylabel("Count")


plt.tight_layout()
plt.show()

# CVD PREVALENCE BY CATEGORICAL FEATURE

fig, axes = plt.subplots(2, 3, figsize=(16, 9))


for ax, feature in zip(axes.flatten(), categorical_features):
    sns.barplot(
        data=viz_df, x=feature, y="cardio", palette="rocket", errorbar=None, ax=ax
    )

    ax.set_title(f"CVD Rate by {feature.capitalize()}")

    ax.set_ylabel("Proportion with CVD")

    ax.set_xlabel("")

    ax.set_ylim(0, 1)


plt.tight_layout()
plt.show()

# CONTINUOUS FEATURE HISTOGRAMS

df[continuous_cols].hist(bins=30, figsize=(12, 8), color="steelblue", edgecolor="black")

plt.suptitle("Histograms of Continuous Features", y=1.02)

plt.tight_layout()
plt.show()

# PEARSON CORRELATION ANALYSIS

corr_matrix = df.corr(numeric_only=True, method="pearson")


plt.figure(figsize=(12, 10))


mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)


sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    square=True,
)


plt.title("Pearson Correlation Heatmap")

plt.tight_layout()
plt.show()

# Correlation with Target

target_corr = corr_matrix["cardio"].drop("cardio").sort_values(key=abs, ascending=False)


plt.figure(figsize=(8, 6))


sns.barplot(x=target_corr.values, y=target_corr.index, palette="coolwarm")


plt.title("Feature Correlation with Target (cardio)")

plt.xlabel("Pearson Correlation Coefficient")

plt.ylabel("Feature")

plt.axvline(0, color="black", linewidth=0.8)

plt.tight_layout()
plt.show()


print("\nTarget Correlations:")

print(target_corr.round(4))

# BOXPLOTS BY CVD STATUS

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

axes = axes.flatten()


for i, feature in enumerate(continuous_cols):
    sns.boxplot(data=viz_df, x="cardio_label", y=feature, palette="Set1", ax=axes[i])

    axes[i].set_title(f"{feature} by CVD Status")

    axes[i].set_xlabel("")
    axes[i].set_ylabel(feature)


# Remove unused sixth subplot
fig.delaxes(axes[-1])


plt.suptitle(
    "Continuous Features by Cardiovascular Disease Status", y=1.02, fontweight="bold"
)

plt.tight_layout()
plt.show()

print("\n" + "=" * 70)
print("VALIDATING PREPROCESSING CONSTRAINTS")
print("=" * 70)


validation_checks = {
    "Height outside [125, 215] cm": ~df["height"].between(125, 215),
    "Weight outside [40, 200] kg": ~df["weight"].between(40, 200),
    "Systolic BP outside [70, 250] mmHg": ~df["ap_hi"].between(70, 250),
    "Diastolic BP outside [40, 150] mmHg": ~df["ap_lo"].between(40, 150),
    "Diastolic BP >= systolic BP": df["ap_lo"] >= df["ap_hi"],
}


for description, mask in validation_checks.items():
    count = int(mask.sum())

    print(f"{description}: {count}")

assert validation_checks["Height outside [125, 215] cm"].sum() == 0

assert validation_checks["Weight outside [40, 200] kg"].sum() == 0

assert validation_checks["Systolic BP outside [70, 250] mmHg"].sum() == 0

assert validation_checks["Diastolic BP outside [40, 150] mmHg"].sum() == 0

assert validation_checks["Diastolic BP >= systolic BP"].sum() == 0


print("\nAll preprocessing constraints were successfully validated.")

print("\n" + "=" * 70)
print("STATISTICAL IQR OUTLIER DIAGNOSTIC")
print("=" * 70)


def iqr_outlier_summary(series):

    q1 = series.quantile(0.25)

    q3 = series.quantile(0.75)

    iqr = q3 - q1

    lower = q1 - 1.5 * iqr

    upper = q3 + 1.5 * iqr

    outlier_mask = (series < lower) | (series > upper)

    return {
        "Count": int(outlier_mask.sum()),
        "Lower Bound": lower,
        "Upper Bound": upper,
    }


iqr_results = []


for col in ["height", "weight", "ap_hi", "ap_lo"]:
    result = iqr_outlier_summary(df[col])

    iqr_results.append(
        {
            "Feature": col,
            "IQR Outliers": result["Count"],
            "Lower Bound": result["Lower Bound"],
            "Upper Bound": result["Upper Bound"],
        }
    )


iqr_summary = pd.DataFrame(iqr_results)


display(iqr_summary.round(2))


print(
    "Note: these observations are statistical IQR outliers only. "
    "They were retained because the preprocessing strategy uses "
    "broad physiological plausibility thresholds rather than "
    "automatic IQR-based removal."
)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))


sns.scatterplot(
    data=viz_df,
    x="weight",
    y="height",
    hue="cardio_label",
    alpha=0.5,
    palette="Set1",
    ax=axes[0],
)


axes[0].set_title("Weight vs Height (by Cardio Risk)")


sns.scatterplot(
    data=viz_df,
    x="ap_hi",
    y="ap_lo",
    hue="cardio_label",
    alpha=0.4,
    palette="Set1",
    ax=axes[1],
)


axes[1].set_title("Systolic vs Diastolic BP (by Cardio Risk)")


plt.tight_layout()
plt.show()

PAIRPLOT_SAMPLE_SIZE = min(3000, len(df))


pairplot_sample = df.sample(n=PAIRPLOT_SAMPLE_SIZE, random_state=42).copy()


pairplot_sample["cardio_label"] = pairplot_sample["cardio"].map(
    {0: "No CVD", 1: "Has CVD"}
)


pair_grid = sns.pairplot(
    pairplot_sample,
    vars=["age_years", "weight", "ap_hi", "ap_lo"],
    hue="cardio_label",
    palette="Set1",
    plot_kws={"alpha": 0.4, "s": 15},
    diag_kind="kde",
)


pair_grid.fig.suptitle(
    (
        "Pairwise Relationships Between Key Risk Factors "
        f"(n={PAIRPLOT_SAMPLE_SIZE:,} sample)"
    ),
    y=1.02,
)


plt.show()

# Feature Engineering

In [ ]:
import os
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")

BASE_PATH = "/content/drive/MyDrive/COMP721 Project"

DATA_PATH = f"{BASE_PATH}/data"
CLEANED_DATA_PATH = f"{DATA_PATH}/cardio_train_cleaned.csv"

if not os.path.exists(CLEANED_DATA_PATH):
    raise FileNotFoundError(f"Cleaned dataset not found at:\n{CLEANED_DATA_PATH}")

df_clean = pd.read_csv(CLEANED_DATA_PATH)

print("Cleaned dataset loaded fresh from Google Drive.")
print(f"Shape: {df_clean.shape[0]:,} rows, {df_clean.shape[1]} columns")

expected_columns = [
    "gender",
    "height",
    "weight",
    "ap_hi",
    "ap_lo",
    "cholesterol",
    "gluc",
    "smoke",
    "alco",
    "active",
    "cardio",
    "age_years",
]

missing_columns = [col for col in expected_columns if col not in df_clean.columns]

unexpected_columns = [col for col in df_clean.columns if col not in expected_columns]

if missing_columns:
    raise ValueError(f"Cleaned dataset is missing expected columns: {missing_columns}")

if unexpected_columns:
    raise ValueError(
        f"Cleaned dataset contains unexpected columns: {unexpected_columns}"
    )

# Validate the corrected preprocessing result
EXPECTED_ROWS = 68517
EXPECTED_COLUMNS = 12

if df_clean.shape != (EXPECTED_ROWS, EXPECTED_COLUMNS):
    raise ValueError(
        "Unexpected cleaned dataset shape.\n"
        f"Expected: ({EXPECTED_ROWS}, {EXPECTED_COLUMNS})\n"
        f"Found:    {df_clean.shape}\n"
        "Check that cardio_train_cleaned.csv was regenerated using "
        "the corrected preprocessing code."
    )

if df_clean.isnull().sum().sum() != 0:
    raise ValueError("Missing values detected in the cleaned dataset.")

if df_clean.duplicated().sum() != 0:
    raise ValueError("Exact duplicate rows detected in the cleaned dataset.")

# Validate preprocessing constraints
if not df_clean["height"].between(125, 215).all():
    raise ValueError("Invalid height values detected.")

if not df_clean["weight"].between(40, 200).all():
    raise ValueError("Invalid weight values detected.")

if not df_clean["ap_hi"].between(70, 250).all():
    raise ValueError("Invalid systolic BP values detected.")

if not df_clean["ap_lo"].between(40, 150).all():
    raise ValueError("Invalid diastolic BP values detected.")

if not (df_clean["ap_hi"] > df_clean["ap_lo"]).all():
    raise ValueError("Rows detected where diastolic BP >= systolic BP.")

print("Cleaned dataset validation passed.")

df_fe = df_clean.copy()

print("Starting Feature Engineering...")

print(f"Input cleaned dataset: {df_fe.shape[0]:,} rows, {df_fe.shape[1]} columns")

# BODY MASS INDEX (BMI)

df_fe["bmi"] = df_fe["weight"] / ((df_fe["height"] / 100.0) ** 2)

# PULSE PRESSURE

df_fe["pulse_pressure"] = df_fe["ap_hi"] - df_fe["ap_lo"]

# MEAN ARTERIAL PRESSURE (MAP)

df_fe["map"] = df_fe["ap_lo"] + (df_fe["pulse_pressure"] / 3.0)

# HYPERTENSION STAGE

htn_conditions = [
    # Stage 2
    ((df_fe["ap_hi"] >= 140) | (df_fe["ap_lo"] >= 90)),
    # Stage 1
    (
        df_fe["ap_hi"].between(130, 139, inclusive="both")
        | df_fe["ap_lo"].between(80, 89, inclusive="both")
    ),
    # Elevated
    (df_fe["ap_hi"].between(120, 129, inclusive="both") & (df_fe["ap_lo"] < 80)),
    # Normal
    ((df_fe["ap_hi"] < 120) & (df_fe["ap_lo"] < 80)),
]

htn_values = [
    3,  # Stage 2
    2,  # Stage 1
    1,  # Elevated
    0,  # Normal
]

df_fe["htn_stage"] = np.select(htn_conditions, htn_values, default=3)

# BMI CATEGORY

bmi_conditions = [
    df_fe["bmi"] < 18.5,
    ((df_fe["bmi"] >= 18.5) & (df_fe["bmi"] < 25.0)),
    ((df_fe["bmi"] >= 25.0) & (df_fe["bmi"] < 30.0)),
    df_fe["bmi"] >= 30.0,
]

bmi_values = [0, 1, 2, 3]

df_fe["bmi_category"] = np.select(bmi_conditions, bmi_values, default=1)

# AGE CATEGORY

df_fe["age_category"] = pd.cut(
    df_fe["age_years"], bins=[0, 40, 50, 60, np.inf], labels=[0, 1, 2, 3], right=False
).astype(int)

# LIFESTYLE RISK SCORE

df_fe["lifestyle_risk"] = (
    df_fe["smoke"].astype(int)
    + df_fe["alco"].astype(int)
    + (1 - df_fe["active"].astype(int))
)

# METABOLIC RISK SCORE

chol_penalty = df_fe["cholesterol"].astype(int) - 1

gluc_penalty = df_fe["gluc"].astype(int) - 1

obesity_penalty = (df_fe["bmi_category"] == 3).astype(int)


df_fe["metabolic_risk"] = chol_penalty + gluc_penalty + obesity_penalty

# CARDIOVASCULAR RISK-FACTOR COUNT

df_fe["cvd_risk_count"] = (
    (df_fe["htn_stage"] >= 2).astype(int)
    + (df_fe["bmi_category"] >= 2).astype(int)
    + (df_fe["smoke"] == 1).astype(int)
    + (df_fe["cholesterol"] > 1).astype(int)
    + (df_fe["gluc"] > 1).astype(int)
    + (df_fe["active"] == 0).astype(int)
)

# METABOLIC RISK FLAG

metabolic_components = (
    (df_fe["bmi_category"] >= 2).astype(int)
    + (df_fe["htn_stage"] >= 2).astype(int)
    + (df_fe["cholesterol"] > 1).astype(int)
)


df_fe["metabolic_risk_flag"] = (metabolic_components >= 2).astype(int)

# AGE-CHOLESTEROL INTERACTION

df_fe["age_cholesterol_interaction"] = df_fe["age_years"] * df_fe["cholesterol"]

df_model = df_fe.drop(columns=["id", "age"], errors="ignore")

engineered_features = [
    "bmi",
    "pulse_pressure",
    "map",
    "htn_stage",
    "bmi_category",
    "age_category",
    "lifestyle_risk",
    "metabolic_risk",
    "cvd_risk_count",
    "metabolic_risk_flag",
    "age_cholesterol_interaction",
]

missing_engineered_features = [
    feature for feature in engineered_features if feature not in df_model.columns
]

if missing_engineered_features:
    raise ValueError(f"Missing engineered features: {missing_engineered_features}")

engineered_numeric = df_model[engineered_features].replace([np.inf, -np.inf], np.nan)

if engineered_numeric.isnull().any().any():
    raise ValueError("NaN or infinite values detected in engineered features.")

assert df_model["htn_stage"].between(0, 3).all()

assert df_model["bmi_category"].between(0, 3).all()

assert df_model["age_category"].between(0, 3).all()

assert df_model["lifestyle_risk"].between(0, 3).all()

assert df_model["metabolic_risk"].between(0, 5).all()

assert df_model["cvd_risk_count"].between(0, 6).all()

assert df_model["metabolic_risk_flag"].isin([0, 1]).all()

stage2_mask = (df_model["ap_hi"] >= 140) | (df_model["ap_lo"] >= 90)

assert (df_model.loc[stage2_mask, "htn_stage"] == 3).all()


normal_mask = (df_model["ap_hi"] < 120) & (df_model["ap_lo"] < 80)

assert (df_model.loc[normal_mask, "htn_stage"] == 0).all()

print(f"\nFeature Engineering complete.")

print(f"Dataset shape: {df_model.shape[0]:,} rows, {df_model.shape[1]} columns")

print(f"Original cleaned variables: {df_clean.shape[1]}")

print(f"Engineered variables created: {len(engineered_features)}")

print("\nNew engineered features:")

for feature in engineered_features:
    print(f"  - {feature}")


print("\nFeature ranges / distributions:")

for feature in [
    "htn_stage",
    "bmi_category",
    "age_category",
    "lifestyle_risk",
    "metabolic_risk",
    "cvd_risk_count",
    "metabolic_risk_flag",
]:
    print(f"\n{feature}:")

    print(df_model[feature].value_counts().sort_index())


print("\nContinuous engineered-feature summary:")

display(
    df_model[["bmi", "pulse_pressure", "map", "age_cholesterol_interaction"]]
    .describe()
    .T.round(2)
)


print(
    "\nNOTE: Scaling is not performed during feature engineering. "
    "Scaling will be fitted using training data only during the "
    "feature-selection/model-development pipeline."
)

BASE_PATH = "/content/drive/MyDrive/COMP721 Project"

FE_PATH = f"{BASE_PATH}/feature_engineering"

os.makedirs(FE_PATH, exist_ok=True)

output_file_fe = f"{FE_PATH}/cardio_train_engineered.csv"

df_model.to_csv(output_file_fe, index=False)

print(f"\nFeature-engineered data successfully saved to:\n{output_file_fe}")

In [ ]:
import os
import warnings
from google.colab import drive
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

drive.mount("/content/drive")

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="muted")

BASE_PATH = "/content/drive/MyDrive/COMP721 Project"

ENGINEERED_DATA_PATH = f"{BASE_PATH}/feature_engineering/cardio_train_engineered.csv"

if not os.path.exists(ENGINEERED_DATA_PATH):
    raise FileNotFoundError(
        f"Feature-engineered dataset not found at:\n"
        f"{ENGINEERED_DATA_PATH}\n\n"
        "Run the corrected Feature Engineering block first."
    )

df_fe = pd.read_csv(ENGINEERED_DATA_PATH)

print("Feature-engineered dataset loaded fresh from Google Drive.")

print(f"Shape: {df_fe.shape[0]:,} rows, {df_fe.shape[1]} columns")

EXPECTED_ROWS = 68517

if len(df_fe) != EXPECTED_ROWS:
    raise ValueError(
        "Unexpected number of rows in the feature-engineered dataset.\n"
        f"Expected: {EXPECTED_ROWS:,}\n"
        f"Found:    {len(df_fe):,}\n\n"
        "The saved feature-engineered file may have been generated "
        "from an older preprocessing run."
    )


required_columns = ["cardio", "bmi", "pulse_pressure", "map", "lifestyle_risk"]

missing_columns = [col for col in required_columns if col not in df_fe.columns]

if missing_columns:
    raise ValueError(
        "Required columns are missing from the "
        f"feature-engineered dataset: {missing_columns}"
    )


if df_fe[required_columns].isnull().any().any():
    raise ValueError("Missing values detected in variables required for Figure 11.")


if not df_fe["cardio"].isin([0, 1]).all():
    raise ValueError("Unexpected values detected in the target variable 'cardio'.")


if not df_fe["lifestyle_risk"].between(0, 3).all():
    raise ValueError(
        "Unexpected values detected in 'lifestyle_risk'. Expected values from 0 to 3."
    )


print("Figure 11 dataset validation passed.")

df_plot = df_fe.copy()

df_plot["cardio_label"] = df_plot["cardio"].map({0: "No CVD (0)", 1: "Has CVD (1)"})

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# BMI BY CVD STATUS

sns.boxplot(data=df_plot, x="cardio_label", y="bmi", ax=axes[0, 0], palette="Set2")

axes[0, 0].set_title("BMI by CVD Status")

axes[0, 0].set_xlabel("")

axes[0, 0].set_ylabel("Body Mass Index (kg/m²)")

# PULSE PRESSURE BY CVD STATUS

sns.boxplot(
    data=df_plot, x="cardio_label", y="pulse_pressure", ax=axes[0, 1], palette="Set2"
)

axes[0, 1].set_title("Pulse Pressure by CVD Status")

axes[0, 1].set_xlabel("")

axes[0, 1].set_ylabel("Pulse Pressure (mmHg)")

# MEAN ARTERIAL PRESSURE BY CVD STATUS

sns.boxplot(data=df_plot, x="cardio_label", y="map", ax=axes[1, 0], palette="Set2")

axes[1, 0].set_title("Mean Arterial Pressure by CVD Status")

axes[1, 0].set_xlabel("")

axes[1, 0].set_ylabel("MAP (mmHg)")

# LIFESTYLE RISK SCORE BY CVD STATUS
sns.countplot(
    data=df_plot, x="lifestyle_risk", hue="cardio_label", ax=axes[1, 1], palette="Set2"
)

axes[1, 1].set_title("Lifestyle Risk Score by CVD Status")

axes[1, 1].set_xlabel("Lifestyle Risk Score")

axes[1, 1].set_ylabel("Patient Count")

axes[1, 1].legend(title="CVD Status")

plt.suptitle(
    "Engineered Features by CVD Status", fontsize=16, fontweight="bold", y=1.02
)

plt.tight_layout()

plt.show()

# Feature Selection

In [ ]:
# Feature Selection

import os
import json
import warnings
from functools import partial

import numpy as np
import pandas as pd

from sklearn.feature_selection import RFE, SelectKBest, f_classif, mutual_info_classif

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

from google.colab import drive

drive.mount("/content/drive")

warnings.filterwarnings("ignore")

BASE_PATH = "/content/drive/MyDrive/COMP721 Project"

ENGINEERED_DATA_PATH = f"{BASE_PATH}/feature_engineering/cardio_train_engineered.csv"

SELECTION_PATH = f"{BASE_PATH}/feature_selection"

os.makedirs(SELECTION_PATH, exist_ok=True)

if not os.path.exists(ENGINEERED_DATA_PATH):
    raise FileNotFoundError(
        f"Feature-engineered dataset not found at:\n"
        f"{ENGINEERED_DATA_PATH}\n\n"
        "Run the corrected Feature Engineering block first."
    )


df_model = pd.read_csv(ENGINEERED_DATA_PATH)

print("Feature-engineered dataset loaded fresh from Google Drive.")

print(f"Shape: {df_model.shape[0]:,} rows, {df_model.shape[1]} columns")

EXPECTED_ROWS = 68517
EXPECTED_COLUMNS = 23


if df_model.shape != (EXPECTED_ROWS, EXPECTED_COLUMNS):
    raise ValueError(
        "Unexpected feature-engineered dataset shape.\n"
        f"Expected: ({EXPECTED_ROWS}, {EXPECTED_COLUMNS})\n"
        f"Found:    {df_model.shape}\n\n"
        "The file may have been generated by an older preprocessing "
        "or feature-engineering run."
    )


expected_engineered_features = [
    "bmi",
    "pulse_pressure",
    "map",
    "htn_stage",
    "bmi_category",
    "age_category",
    "lifestyle_risk",
    "metabolic_risk",
    "cvd_risk_count",
    "metabolic_risk_flag",
    "age_cholesterol_interaction",
]


missing_engineered = [
    feature
    for feature in expected_engineered_features
    if feature not in df_model.columns
]


if missing_engineered:
    raise ValueError(f"Missing expected engineered features:\n{missing_engineered}")


if df_model.isnull().sum().sum() != 0:
    raise ValueError("Missing values detected in feature-engineered dataset.")


if not df_model["cardio"].isin([0, 1]).all():
    raise ValueError("Unexpected target values detected in 'cardio'.")


print("Feature-engineered dataset validation passed.")

original_features = [
    "gender",
    "height",
    "weight",
    "ap_hi",
    "ap_lo",
    "cholesterol",
    "gluc",
    "smoke",
    "alco",
    "active",
    "age_years",
]


engineered_features = [
    "bmi",
    "pulse_pressure",
    "map",
    "htn_stage",
    "bmi_category",
    "age_category",
    "lifestyle_risk",
    "metabolic_risk",
    "cvd_risk_count",
    "metabolic_risk_flag",
    "age_cholesterol_interaction",
]


all_features = original_features + engineered_features


if len(set(all_features)) != len(all_features):
    raise ValueError("Duplicate feature names detected.")


missing_features = [
    feature for feature in all_features if feature not in df_model.columns
]


if missing_features:
    raise ValueError(f"Required features missing from dataset: {missing_features}")

print("\nExecuting canonical 80/20 stratified train/test split...")


X = df_model[all_features].copy()

y = df_model["cardio"].copy()


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)


print(f"Training Set: {X_train.shape[0]:,} rows")

print(f"Test Set: {X_test.shape[0]:,} rows")


print("\nTraining target distribution:")

print(y_train.value_counts(normalize=True).sort_index().mul(100).round(2))


print("\nTest target distribution:")

print(y_test.value_counts(normalize=True).sort_index().mul(100).round(2))

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


reference_tree = DecisionTreeClassifier(max_depth=5, random_state=42)

print("\n" + "=" * 75)

print("FEATURE SUBSET COMPARISON (5-Fold CV on Training Data Only)")

print("=" * 75)


def evaluate_fixed_subset(features, subset_name):

    scores = cross_val_score(
        estimator=reference_tree,
        X=X_train[features],
        y=y_train,
        cv=cv,
        scoring="accuracy",
        n_jobs=-1,
    )

    print(
        f"{subset_name:<38} "
        f"Accuracy: "
        f"{scores.mean() * 100:.2f}% "
        f"(+/- {scores.std() * 100:.2f}%)"
    )

    return scores


original_scores = evaluate_fixed_subset(original_features, "1. Original Features Only")


engineered_scores = evaluate_fixed_subset(
    engineered_features, "2. Engineered Features Only"
)


all_scores = evaluate_fixed_subset(
    all_features, "3. All Features (Original + Engineered)"
)

# ANOVA SelectKBest

anova_pipeline = Pipeline(
    [
        ("selector", SelectKBest(score_func=f_classif, k=5)),
        ("classifier", DecisionTreeClassifier(max_depth=5, random_state=42)),
    ]
)


anova_cv_scores = cross_val_score(
    estimator=anova_pipeline, X=X_train, y=y_train, cv=cv, scoring="accuracy", n_jobs=-1
)


print(
    f"{'4. Selected: Top 5 ANOVA':<38} "
    f"Accuracy: "
    f"{anova_cv_scores.mean() * 100:.2f}% "
    f"(+/- {anova_cv_scores.std() * 100:.2f}%)"
)

# Mutual Information SelectKBest

mi_score_function = partial(mutual_info_classif, random_state=42)


mi_pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("selector", SelectKBest(score_func=mi_score_function, k=5)),
        ("classifier", DecisionTreeClassifier(max_depth=5, random_state=42)),
    ]
)


mi_cv_scores = cross_val_score(
    estimator=mi_pipeline, X=X_train, y=y_train, cv=cv, scoring="accuracy", n_jobs=-1
)


print(
    f"{'5. Selected: Top 5 Mutual Information':<38} "
    f"Accuracy: "
    f"{mi_cv_scores.mean() * 100:.2f}% "
    f"(+/- {mi_cv_scores.std() * 100:.2f}%)"
)

# Recursive Feature Elimination

rfe_pipeline = Pipeline(
    [
        (
            "selector",
            RFE(
                estimator=DecisionTreeClassifier(max_depth=5, random_state=42),
                n_features_to_select=5,
                step=1,
            ),
        ),
        ("classifier", DecisionTreeClassifier(max_depth=5, random_state=42)),
    ]
)


rfe_cv_scores = cross_val_score(
    estimator=rfe_pipeline, X=X_train, y=y_train, cv=cv, scoring="accuracy", n_jobs=-1
)


print(
    f"{'6. Selected: Top 5 RFE':<38} "
    f"Accuracy: "
    f"{rfe_cv_scores.mean() * 100:.2f}% "
    f"(+/- {rfe_cv_scores.std() * 100:.2f}%)"
)

# Fit each selector on the complete training set

# ANOVA
anova_selector_final = SelectKBest(score_func=f_classif, k=5)

anova_selector_final.fit(X_train, y_train)


anova_scores = pd.Series(
    anova_selector_final.scores_, index=X_train.columns
).sort_values(ascending=False)


top5_anova = X_train.columns[anova_selector_final.get_support()].tolist()


print("\n--- Top 5 Features (ANOVA F-Score; fitted on full training set) ---")

print(anova_scores.head(5).round(2))

# Mutual Information

# Fit scaler using training data only
mi_scaler_final = StandardScaler()

X_train_mi_scaled = mi_scaler_final.fit_transform(X_train)


mi_selector_final = SelectKBest(score_func=mi_score_function, k=5)

mi_selector_final.fit(X_train_mi_scaled, y_train)


mi_scores = pd.Series(mi_selector_final.scores_, index=X_train.columns).sort_values(
    ascending=False
)


top5_mi = X_train.columns[mi_selector_final.get_support()].tolist()


print("\n--- Top 5 Features (Mutual Information; fitted on full training set) ---")

print(mi_scores.head(5).round(4))

# RFE
rfe_selector_final = RFE(
    estimator=DecisionTreeClassifier(max_depth=5, random_state=42),
    n_features_to_select=5,
    step=1,
)


rfe_selector_final.fit(X_train, y_train)


rfe_selected_mask = rfe_selector_final.get_support()


rfe_selected_features = X_train.columns[rfe_selected_mask].tolist()


rfe_importances = rfe_selector_final.estimator_.feature_importances_


top5_rfe_df = pd.DataFrame(
    {"Feature": rfe_selected_features, "Tree_Importance": rfe_importances}
)


top5_rfe_df = top5_rfe_df.sort_values(
    by="Tree_Importance", ascending=False
).reset_index(drop=True)


rfe_features = top5_rfe_df["Feature"].tolist()


print("\n--- Top 5 Features (RFE; fitted on full training set) ---")

print(top5_rfe_df.to_string(index=False))

subset_candidates = {
    "ANOVA": {"features": top5_anova, "scores": anova_cv_scores},
    "Mutual Information": {"features": top5_mi, "scores": mi_cv_scores},
    "RFE": {"features": rfe_features, "scores": rfe_cv_scores},
}


best_method = max(
    subset_candidates, key=lambda method: subset_candidates[method]["scores"].mean()
)


best_features = subset_candidates[best_method]["features"]


best_scores = subset_candidates[best_method]["scores"]


best_acc = best_scores.mean() * 100


print("\n" + "=" * 75)

print("FINAL FEATURE-SELECTION RESULT")

print("=" * 75)


print(f"Selected method: {best_method}")


print(f"Mean CV Accuracy: {best_acc:.2f}%")


print(f"CV Standard Deviation: {best_scores.std() * 100:.2f}%")


print("\nSelected features:")


for feature in best_features:
    print(f"  - {feature}")

train_final = X_train[best_features].copy()


train_final["cardio"] = y_train.values


test_final = X_test[best_features].copy()


test_final["cardio"] = y_test.values

train_file = f"{SELECTION_PATH}/train_final.csv"

test_file = f"{SELECTION_PATH}/test_final.csv"


train_final.to_csv(train_file, index=False)


test_final.to_csv(test_file, index=False)


print(f"\nFinal UN-SCALED training subset saved to:\n{train_file}")


print(f"\nFinal UN-SCALED test subset saved to:\n{test_file}")

selection_summary = {
    "random_state": 42,
    "train_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
    "selected_method": best_method,
    "selected_features": best_features,
    "cv_folds": 5,
    "cv_scoring": "accuracy",
    "original_features_cv_accuracy_mean": float(original_scores.mean()),
    "engineered_features_cv_accuracy_mean": float(engineered_scores.mean()),
    "all_features_cv_accuracy_mean": float(all_scores.mean()),
    "anova_cv_accuracy_mean": float(anova_cv_scores.mean()),
    "anova_cv_accuracy_std": float(anova_cv_scores.std()),
    "mi_cv_accuracy_mean": float(mi_cv_scores.mean()),
    "mi_cv_accuracy_std": float(mi_cv_scores.std()),
    "rfe_cv_accuracy_mean": float(rfe_cv_scores.mean()),
    "rfe_cv_accuracy_std": float(rfe_cv_scores.std()),
    "saved_data_scaled": False,
}


metadata_file = f"{SELECTION_PATH}/feature_selection_summary.json"


with open(metadata_file, "w") as f:
    json.dump(selection_summary, f, indent=4)


print(f"\nFeature-selection metadata saved to:\n{metadata_file}")

print("\n" + "=" * 75)

print("FINAL DATASET SANITY CHECK")

print("=" * 75)


print(f"Training rows: {len(train_final):,}")

print(f"Test rows: {len(test_final):,}")

print(f"Predictor count: {len(best_features)}")

print(f"Training missing values: {train_final.isnull().sum().sum()}")

print(f"Test missing values: {test_final.isnull().sum().sum()}")

print("\nFeature selection completed without using the held-out test set.")

# Machine Learning Models

## Decision Tree

In [ ]:
# DECISION TREE

import os
import json
import warnings
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, cross_val_score

from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    matthews_corrcoef,
    cohen_kappa_score,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
    average_precision_score,
)

from google.colab import drive

drive.mount("/content/drive")

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="muted")

BASE_PATH = "/content/drive/MyDrive/COMP721 Project"

SELECTION_PATH = f"{BASE_PATH}/feature_selection"

MODEL_PATH = f"{BASE_PATH}/models"

RESULTS_PATH = f"{BASE_PATH}/model_results/decision_tree"

os.makedirs(MODEL_PATH, exist_ok=True)

os.makedirs(RESULTS_PATH, exist_ok=True)


TRAIN_FILE = f"{SELECTION_PATH}/train_final.csv"

TEST_FILE = f"{SELECTION_PATH}/test_final.csv"

SELECTION_METADATA_FILE = f"{SELECTION_PATH}/feature_selection_summary.json"

required_files = [TRAIN_FILE, TEST_FILE, SELECTION_METADATA_FILE]

for file_path in required_files:
    if not os.path.exists(file_path):
        raise FileNotFoundError(
            f"Required file not found:\n{file_path}\n\n"
            "Run the corrected Feature Selection block first."
        )


train_df = pd.read_csv(TRAIN_FILE)

test_df = pd.read_csv(TEST_FILE)


with open(SELECTION_METADATA_FILE, "r") as f:
    selection_metadata = json.load(f)


print("Feature-selection outputs loaded fresh from Google Drive.")

expected_features = selection_metadata["selected_features"]

expected_train_rows = selection_metadata["train_rows"]

expected_test_rows = selection_metadata["test_rows"]


expected_columns = expected_features + ["cardio"]


if train_df.shape[0] != expected_train_rows:
    raise ValueError(
        "Unexpected training-set size.\n"
        f"Expected: {expected_train_rows:,}\n"
        f"Found:    {train_df.shape[0]:,}"
    )


if test_df.shape[0] != expected_test_rows:
    raise ValueError(
        "Unexpected test-set size.\n"
        f"Expected: {expected_test_rows:,}\n"
        f"Found:    {test_df.shape[0]:,}"
    )


if list(train_df.columns) != expected_columns:
    raise ValueError(
        "Training columns do not match the feature-selection output.\n"
        f"Expected: {expected_columns}\n"
        f"Found:    {train_df.columns.tolist()}"
    )


if list(test_df.columns) != expected_columns:
    raise ValueError(
        "Test columns do not match the feature-selection output.\n"
        f"Expected: {expected_columns}\n"
        f"Found:    {test_df.columns.tolist()}"
    )


if train_df.isnull().sum().sum() != 0:
    raise ValueError("Missing values detected in training data.")


if test_df.isnull().sum().sum() != 0:
    raise ValueError("Missing values detected in test data.")


if not train_df["cardio"].isin([0, 1]).all():
    raise ValueError("Unexpected target values in training data.")


if not test_df["cardio"].isin([0, 1]).all():
    raise ValueError("Unexpected target values in test data.")


print("Decision Tree input validation passed.")

X_train = train_df[expected_features].copy()

y_train = train_df["cardio"].copy()


X_test = test_df[expected_features].copy()

y_test = test_df["cardio"].copy()


print(f"\nTraining Set: {X_train.shape[0]:,} rows")

print(f"Test Set (held out): {X_test.shape[0]:,} rows")

print(f"Predictors: {X_train.shape[1]}")

print("\nPredictor variables:")

for feature in X_train.columns:
    print(f"  - {feature}")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("\n" + "=" * 75)

print("DECISION TREE: 5-FOLD CROSS-VALIDATION")

print("=" * 75)


depths = [5, 10, 15, 20]


cv_results = {}


for depth in depths:
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)

    scores = cross_val_score(
        estimator=model, X=X_train, y=y_train, cv=cv, scoring="accuracy", n_jobs=-1
    )

    cv_results[depth] = {"scores": scores, "mean": scores.mean(), "std": scores.std()}

    print(
        f"max_depth={depth:<2} | "
        f"Accuracy: "
        f"{scores.mean() * 100:.2f}% "
        f"(+/- {scores.std() * 100:.2f}%)"
    )

best_depth = max(cv_results, key=lambda depth: cv_results[depth]["mean"])


best_cv_scores = cv_results[best_depth]["scores"]


best_cv_mean = cv_results[best_depth]["mean"]


best_cv_std = cv_results[best_depth]["std"]


print("\nBest Decision Tree configuration:")

print(f"  max_depth = {best_depth}")

print(f"  Mean CV Accuracy = {best_cv_mean * 100:.2f}%")

print(f"  CV Standard Deviation = {best_cv_std * 100:.2f}%")

best_dt = DecisionTreeClassifier(max_depth=best_depth, random_state=42)


best_dt.fit(X_train, y_train)


print("\nSelected Decision Tree retrained on the complete training partition.")

DT_MODEL_FILE = f"{MODEL_PATH}/decision_tree_final.joblib"


joblib.dump(best_dt, DT_MODEL_FILE)


print(f"\nTrained Decision Tree saved to:\n{DT_MODEL_FILE}")

DT_CV_SCORES_FILE = f"{RESULTS_PATH}/decision_tree_cv_fold_scores.npy"


np.save(DT_CV_SCORES_FILE, best_cv_scores)


print(f"\nDecision Tree CV fold scores saved to:\n{DT_CV_SCORES_FILE}")

# Feature Importance
importances = best_dt.feature_importances_


feat_imp_df = pd.DataFrame({"Feature": X_train.columns, "Importance": importances})


feat_imp_df = feat_imp_df.sort_values(by="Importance", ascending=False).reset_index(
    drop=True
)


print("\n" + "=" * 75)

print("DECISION TREE FEATURE IMPORTANCE")

print("=" * 75)


for _, row in feat_imp_df.iterrows():
    print(f"{row['Feature']:<32} {row['Importance'] * 100:>6.2f}%")


plt.figure(figsize=(10, 6))


sns.barplot(data=feat_imp_df, x="Importance", y="Feature", palette="viridis")


plt.title("Decision Tree Feature Importances")

plt.xlabel("Gini Importance")

plt.ylabel("Feature")

plt.tight_layout()
plt.show()


# Save importance values for later reporting
FEATURE_IMPORTANCE_FILE = f"{RESULTS_PATH}/decision_tree_feature_importance.csv"


feat_imp_df.to_csv(FEATURE_IMPORTANCE_FILE, index=False)

print("\nEvaluating final Decision Tree on the held-out test set...")


y_pred = best_dt.predict(X_test)


y_prob = best_dt.predict_proba(X_test)[:, 1]

cm = confusion_matrix(y_test, y_pred, labels=[0, 1])


tn, fp, fn, tp = cm.ravel()


specificity = tn / (tn + fp)


accuracy = accuracy_score(y_test, y_pred)


precision = precision_score(y_test, y_pred, zero_division=0)


recall = recall_score(y_test, y_pred, zero_division=0)


f1 = f1_score(y_test, y_pred, zero_division=0)


roc_auc = roc_auc_score(y_test, y_prob)


mcc = matthews_corrcoef(y_test, y_pred)


kappa = cohen_kappa_score(y_test, y_pred)


average_precision = average_precision_score(y_test, y_prob)

print("\n" + "=" * 75)

print("FINAL TEST SET PERFORMANCE — DECISION TREE")

print("=" * 75)


print(f"Accuracy:          {accuracy * 100:.2f}%")

print(f"Precision:         {precision * 100:.2f}%")

print(f"Recall:            {recall * 100:.2f}%")

print(f"Specificity:       {specificity * 100:.2f}%")

print(f"F1-Score:          {f1 * 100:.2f}%")

print(f"ROC-AUC:           {roc_auc:.4f}")

print(f"Average Precision: {average_precision:.4f}")

print(f"MCC:               {mcc:.4f}")

print(f"Cohen's Kappa:     {kappa:.4f}")


print("\nConfusion Matrix Values:")

print(f"TN = {tn:,}")

print(f"FP = {fp:,}")

print(f"FN = {fn:,}")

print(f"TP = {tp:,}")

prediction_df = pd.DataFrame(
    {"y_true": y_test.to_numpy(), "y_pred": y_pred, "y_prob": y_prob}
)


DT_PREDICTIONS_FILE = f"{RESULTS_PATH}/decision_tree_test_predictions.csv"


prediction_df.to_csv(DT_PREDICTIONS_FILE, index=False)


print(f"\nDecision Tree test predictions saved to:\n{DT_PREDICTIONS_FILE}")

cv_configuration_results = {}


for depth in depths:
    cv_configuration_results[str(depth)] = {
        "fold_scores": cv_results[depth]["scores"].tolist(),
        "mean_accuracy": float(cv_results[depth]["mean"]),
        "std_accuracy": float(cv_results[depth]["std"]),
    }


decision_tree_summary = {
    "model": "DecisionTreeClassifier",
    "random_state": 42,
    "cv_method": "StratifiedKFold",
    "cv_folds": 5,
    "cv_shuffle": True,
    "cv_random_state": 42,
    "hyperparameter": "max_depth",
    "tested_depths": depths,
    "cv_results": cv_configuration_results,
    "best_depth": int(best_depth),
    "best_cv_accuracy_mean": float(best_cv_mean),
    "best_cv_accuracy_std": float(best_cv_std),
    "selected_features": X_train.columns.tolist(),
    "training_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
    "test_metrics": {
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "specificity": float(specificity),
        "f1_score": float(f1),
        "roc_auc": float(roc_auc),
        "average_precision": float(average_precision),
        "mcc": float(mcc),
        "cohen_kappa": float(kappa),
    },
    "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
    "model_file": DT_MODEL_FILE,
    "cv_scores_file": DT_CV_SCORES_FILE,
    "test_predictions_file": DT_PREDICTIONS_FILE,
}


DT_SUMMARY_FILE = f"{RESULTS_PATH}/decision_tree_summary.json"


with open(DT_SUMMARY_FILE, "w") as f:
    json.dump(decision_tree_summary, f, indent=4)


print(f"\nDecision Tree summary saved to:\n{DT_SUMMARY_FILE}")

plt.figure(figsize=(6, 5))


sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    xticklabels=["No CVD (0)", "Has CVD (1)"],
    yticklabels=["No CVD (0)", "Has CVD (1)"],
)


plt.title("Decision Tree Confusion Matrix (Test Set)")

plt.xlabel("Predicted Label")

plt.ylabel("True Label")

plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

fpr, tpr, _ = roc_curve(y_test, y_prob)


axes[0].plot(
    fpr, tpr, color="darkorange", lw=2, label=(f"ROC curve (AUC = {roc_auc:.4f})")
)


axes[0].plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")


axes[0].set_xlim([0.0, 1.0])

axes[0].set_ylim([0.0, 1.05])

axes[0].set_xlabel("False Positive Rate (1 - Specificity)")

axes[0].set_ylabel("True Positive Rate (Recall/Sensitivity)")

axes[0].set_title("Receiver Operating Characteristic (ROC) Curve")

axes[0].legend(loc="lower right")

precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_prob)


axes[1].plot(
    recall_curve,
    precision_curve,
    color="purple",
    lw=2,
    label=(f"PR curve (AP = {average_precision:.4f})"),
)


axes[1].set_xlabel("Recall (Sensitivity)")

axes[1].set_ylabel("Precision (Positive Predictive Value)")

axes[1].set_title("Precision-Recall Curve")

axes[1].legend(loc="lower left")


plt.tight_layout()
plt.show()

print("\n" + "=" * 75)

print("DECISION TREE ARTIFACTS SAVED")

print("=" * 75)


print(f"Model:\n  {DT_MODEL_FILE}")

print(f"\nCV fold scores:\n  {DT_CV_SCORES_FILE}")

print(f"\nTest predictions:\n  {DT_PREDICTIONS_FILE}")

print(f"\nModel/results summary:\n  {DT_SUMMARY_FILE}")

print(f"\nFeature importances:\n  {FEATURE_IMPORTANCE_FILE}")

print("\nDecision Tree training and final evaluation complete.")

## SVM

In [ ]:
# SUPPORT VECTOR MACHINE (SVM)

import os
import json
import warnings
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, cross_val_score

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    matthews_corrcoef,
    cohen_kappa_score,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
    average_precision_score,
)

from google.colab import drive

drive.mount("/content/drive")

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="muted")

BASE_PATH = "/content/drive/MyDrive/COMP721 Project"

SELECTION_PATH = f"{BASE_PATH}/feature_selection"

MODEL_PATH = f"{BASE_PATH}/models"

RESULTS_PATH = f"{BASE_PATH}/model_results/svm"

os.makedirs(MODEL_PATH, exist_ok=True)

os.makedirs(RESULTS_PATH, exist_ok=True)


TRAIN_FILE = f"{SELECTION_PATH}/train_final.csv"

TEST_FILE = f"{SELECTION_PATH}/test_final.csv"

SELECTION_METADATA_FILE = f"{SELECTION_PATH}/feature_selection_summary.json"

required_files = [TRAIN_FILE, TEST_FILE, SELECTION_METADATA_FILE]

for file_path in required_files:
    if not os.path.exists(file_path):
        raise FileNotFoundError(
            f"Required file not found:\n{file_path}\n\n"
            "Run the corrected Feature Selection block first."
        )


train_df = pd.read_csv(TRAIN_FILE)

test_df = pd.read_csv(TEST_FILE)


with open(SELECTION_METADATA_FILE, "r") as f:
    selection_metadata = json.load(f)


print("Feature-selection outputs loaded fresh from Google Drive.")

expected_features = selection_metadata["selected_features"]

expected_train_rows = selection_metadata["train_rows"]

expected_test_rows = selection_metadata["test_rows"]


expected_columns = expected_features + ["cardio"]


if train_df.shape[0] != expected_train_rows:
    raise ValueError(
        "Unexpected training-set size.\n"
        f"Expected: {expected_train_rows:,}\n"
        f"Found:    {train_df.shape[0]:,}"
    )


if test_df.shape[0] != expected_test_rows:
    raise ValueError(
        "Unexpected test-set size.\n"
        f"Expected: {expected_test_rows:,}\n"
        f"Found:    {test_df.shape[0]:,}"
    )


if list(train_df.columns) != expected_columns:
    raise ValueError(
        "Training columns do not match Feature Selection output.\n"
        f"Expected: {expected_columns}\n"
        f"Found:    {train_df.columns.tolist()}"
    )


if list(test_df.columns) != expected_columns:
    raise ValueError(
        "Test columns do not match Feature Selection output.\n"
        f"Expected: {expected_columns}\n"
        f"Found:    {test_df.columns.tolist()}"
    )


if train_df.isnull().sum().sum() != 0:
    raise ValueError("Missing values detected in training data.")


if test_df.isnull().sum().sum() != 0:
    raise ValueError("Missing values detected in test data.")


if not train_df["cardio"].isin([0, 1]).all():
    raise ValueError("Unexpected target values in training data.")


if not test_df["cardio"].isin([0, 1]).all():
    raise ValueError("Unexpected target values in test data.")


print("SVM input validation passed.")

X_train = train_df[expected_features].copy()

y_train = train_df["cardio"].copy()


X_test = test_df[expected_features].copy()

y_test = test_df["cardio"].copy()


print(f"\nTraining Set: {X_train.shape[0]:,} rows")

print(f"Test Set (held out): {X_test.shape[0]:,} rows")

print(f"Predictors: {X_train.shape[1]}")


print("\nPredictor variables:")

for feature in X_train.columns:
    print(f"  - {feature}")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("\n" + "=" * 75)

print("SVM: 5-FOLD CROSS-VALIDATION")

print("=" * 75)


print("Evaluating all four kernels on the complete training partition.")

print("StandardScaler is fitted independently inside each CV training fold.")

print(
    "This stage may take substantially longer than the previous 8,000-row "
    "subsample approach.\n"
)


kernels = ["linear", "poly", "rbf", "sigmoid"]


cv_results = {}


for kernel in kernels:
    svm_pipeline = Pipeline(
        [
            ("scaler", StandardScaler()),
            ("svm", SVC(kernel=kernel, probability=False, random_state=42)),
        ]
    )

    scores = cross_val_score(
        estimator=svm_pipeline,
        X=X_train,
        y=y_train,
        cv=cv,
        scoring="accuracy",
        n_jobs=-1,
    )

    cv_results[kernel] = {"scores": scores, "mean": scores.mean(), "std": scores.std()}

    print(
        f"kernel='{kernel:<7}' | "
        f"Accuracy: "
        f"{scores.mean() * 100:.2f}% "
        f"(+/- {scores.std() * 100:.2f}%)"
    )

best_kernel = max(cv_results, key=lambda kernel: cv_results[kernel]["mean"])


best_cv_scores = cv_results[best_kernel]["scores"]


best_cv_mean = cv_results[best_kernel]["mean"]


best_cv_std = cv_results[best_kernel]["std"]


print("\nBest SVM configuration:")

print(f"  kernel = '{best_kernel}'")

print(f"  Mean CV Accuracy = {best_cv_mean * 100:.2f}%")

print(f"  CV Standard Deviation = {best_cv_std * 100:.2f}%")

print("\nRetraining selected SVM pipeline on the complete training partition.")

print("Probability estimation is enabled for final evaluation and Soft Voting.")

print("This final fit may take several minutes.")


best_svm_pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("svm", SVC(kernel=best_kernel, probability=True, random_state=42)),
    ]
)


best_svm_pipeline.fit(X_train, y_train)


print("\nSelected SVM pipeline retrained on the complete training partition.")

SVM_MODEL_FILE = f"{MODEL_PATH}/svm_final_pipeline.joblib"


joblib.dump(best_svm_pipeline, SVM_MODEL_FILE)


print(f"\nTrained SVM pipeline saved to:\n{SVM_MODEL_FILE}")

SVM_CV_SCORES_FILE = f"{RESULTS_PATH}/svm_cv_fold_scores.npy"


np.save(SVM_CV_SCORES_FILE, best_cv_scores)


print(f"\nSVM CV fold scores saved to:\n{SVM_CV_SCORES_FILE}")

print("\nEvaluating final SVM pipeline on the held-out test set...")


y_pred = best_svm_pipeline.predict(X_test)


y_prob = best_svm_pipeline.predict_proba(X_test)[:, 1]

cm = confusion_matrix(y_test, y_pred, labels=[0, 1])


tn, fp, fn, tp = cm.ravel()


specificity = tn / (tn + fp)


accuracy = accuracy_score(y_test, y_pred)


precision = precision_score(y_test, y_pred, zero_division=0)


recall = recall_score(y_test, y_pred, zero_division=0)


f1 = f1_score(y_test, y_pred, zero_division=0)


roc_auc = roc_auc_score(y_test, y_prob)


mcc = matthews_corrcoef(y_test, y_pred)


kappa = cohen_kappa_score(y_test, y_pred)


average_precision = average_precision_score(y_test, y_prob)

print("\n" + "=" * 75)

print("FINAL TEST SET PERFORMANCE — SVM")

print("=" * 75)


print(f"Accuracy:          {accuracy * 100:.2f}%")

print(f"Precision:         {precision * 100:.2f}%")

print(f"Recall:            {recall * 100:.2f}%")

print(f"Specificity:       {specificity * 100:.2f}%")

print(f"F1-Score:          {f1 * 100:.2f}%")

print(f"ROC-AUC:           {roc_auc:.4f}")

print(f"Average Precision: {average_precision:.4f}")

print(f"MCC:               {mcc:.4f}")

print(f"Cohen's Kappa:     {kappa:.4f}")


print("\nConfusion Matrix Values:")

print(f"TN = {tn:,}")

print(f"FP = {fp:,}")

print(f"FN = {fn:,}")

print(f"TP = {tp:,}")

prediction_df = pd.DataFrame(
    {"y_true": y_test.to_numpy(), "y_pred": y_pred, "y_prob": y_prob}
)


SVM_PREDICTIONS_FILE = f"{RESULTS_PATH}/svm_test_predictions.csv"


prediction_df.to_csv(SVM_PREDICTIONS_FILE, index=False)


print(f"\nSVM test predictions saved to:\n{SVM_PREDICTIONS_FILE}")

cv_configuration_results = {}


for kernel in kernels:
    cv_configuration_results[kernel] = {
        "fold_scores": cv_results[kernel]["scores"].tolist(),
        "mean_accuracy": float(cv_results[kernel]["mean"]),
        "std_accuracy": float(cv_results[kernel]["std"]),
    }


svm_summary = {
    "model": "StandardScaler + SVC",
    "random_state": 42,
    "cv_method": "StratifiedKFold",
    "cv_folds": 5,
    "cv_shuffle": True,
    "cv_random_state": 42,
    "hyperparameter": "kernel",
    "tested_kernels": kernels,
    "cv_results": cv_configuration_results,
    "best_kernel": best_kernel,
    "best_cv_accuracy_mean": float(best_cv_mean),
    "best_cv_accuracy_std": float(best_cv_std),
    "probability_enabled_final_model": True,
    "selected_features": X_train.columns.tolist(),
    "training_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
    "test_metrics": {
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "specificity": float(specificity),
        "f1_score": float(f1),
        "roc_auc": float(roc_auc),
        "average_precision": float(average_precision),
        "mcc": float(mcc),
        "cohen_kappa": float(kappa),
    },
    "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
    "model_file": SVM_MODEL_FILE,
    "cv_scores_file": SVM_CV_SCORES_FILE,
    "test_predictions_file": SVM_PREDICTIONS_FILE,
}


SVM_SUMMARY_FILE = f"{RESULTS_PATH}/svm_summary.json"


with open(SVM_SUMMARY_FILE, "w") as f:
    json.dump(svm_summary, f, indent=4)


print(f"\nSVM summary saved to:\n{SVM_SUMMARY_FILE}")

plt.figure(figsize=(6, 5))


sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    xticklabels=["No CVD (0)", "Has CVD (1)"],
    yticklabels=["No CVD (0)", "Has CVD (1)"],
)


plt.title("SVM Confusion Matrix (Test Set)")

plt.xlabel("Predicted Label")

plt.ylabel("True Label")

plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

fpr, tpr, _ = roc_curve(y_test, y_prob)


axes[0].plot(
    fpr, tpr, color="darkorange", lw=2, label=(f"ROC curve (AUC = {roc_auc:.4f})")
)


axes[0].plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")


axes[0].set_xlim([0.0, 1.0])

axes[0].set_ylim([0.0, 1.05])

axes[0].set_xlabel("False Positive Rate (1 - Specificity)")

axes[0].set_ylabel("True Positive Rate (Recall/Sensitivity)")

axes[0].set_title("Receiver Operating Characteristic (ROC) Curve")

axes[0].legend(loc="lower right")

precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_prob)


axes[1].plot(
    recall_curve,
    precision_curve,
    color="purple",
    lw=2,
    label=(f"PR curve (AP = {average_precision:.4f})"),
)


axes[1].set_xlabel("Recall (Sensitivity)")

axes[1].set_ylabel("Precision (Positive Predictive Value)")

axes[1].set_title("Precision-Recall Curve")

axes[1].legend(loc="lower left")


plt.tight_layout()
plt.show()

print("\n" + "=" * 75)

print("SVM ARTIFACTS SAVED")

print("=" * 75)


print(f"Model pipeline:\n  {SVM_MODEL_FILE}")

print(f"\nCV fold scores:\n  {SVM_CV_SCORES_FILE}")

print(f"\nTest predictions:\n  {SVM_PREDICTIONS_FILE}")

print(f"\nModel/results summary:\n  {SVM_SUMMARY_FILE}")


print("\nSVM training and final evaluation complete.")

# Ensemble Models

## Soft Voting Classifier

In [ ]:
# SOFT VOTING ENSEMBLE

import os
import json
import warnings
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    matthews_corrcoef,
    cohen_kappa_score,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
    average_precision_score,
)

from google.colab import drive

drive.mount("/content/drive")

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="muted")

BASE_PATH = "/content/drive/MyDrive/COMP721 Project"

SELECTION_PATH = f"{BASE_PATH}/feature_selection"

MODEL_PATH = f"{BASE_PATH}/models"

DT_RESULTS_PATH = f"{BASE_PATH}/model_results/decision_tree"

SVM_RESULTS_PATH = f"{BASE_PATH}/model_results/svm"

ENSEMBLE_RESULTS_PATH = f"{BASE_PATH}/model_results/soft_voting"


os.makedirs(MODEL_PATH, exist_ok=True)

os.makedirs(ENSEMBLE_RESULTS_PATH, exist_ok=True)

TRAIN_FILE = f"{SELECTION_PATH}/train_final.csv"

TEST_FILE = f"{SELECTION_PATH}/test_final.csv"

SELECTION_METADATA_FILE = f"{SELECTION_PATH}/feature_selection_summary.json"

DT_MODEL_FILE = f"{MODEL_PATH}/decision_tree_final.joblib"

SVM_MODEL_FILE = f"{MODEL_PATH}/svm_final_pipeline.joblib"

DT_SUMMARY_FILE = f"{DT_RESULTS_PATH}/decision_tree_summary.json"

SVM_SUMMARY_FILE = f"{SVM_RESULTS_PATH}/svm_summary.json"

DT_CV_SCORES_FILE = f"{DT_RESULTS_PATH}/decision_tree_cv_fold_scores.npy"

SVM_CV_SCORES_FILE = f"{SVM_RESULTS_PATH}/svm_cv_fold_scores.npy"

required_files = [
    TRAIN_FILE,
    TEST_FILE,
    SELECTION_METADATA_FILE,
    DT_MODEL_FILE,
    SVM_MODEL_FILE,
    DT_SUMMARY_FILE,
    SVM_SUMMARY_FILE,
    DT_CV_SCORES_FILE,
    SVM_CV_SCORES_FILE,
]


missing_files = [
    file_path for file_path in required_files if not os.path.exists(file_path)
]


if missing_files:
    raise FileNotFoundError(
        "The following required files are missing:\n\n"
        + "\n".join(missing_files)
        + "\n\nEnsure both the corrected Decision Tree and SVM "
        "cells completed successfully before running Soft Voting."
    )


print("All required Decision Tree, SVM, and Feature Selection artifacts were found.")

train_df = pd.read_csv(TRAIN_FILE)

test_df = pd.read_csv(TEST_FILE)


with open(SELECTION_METADATA_FILE, "r") as f:
    selection_metadata = json.load(f)


expected_features = selection_metadata["selected_features"]

expected_train_rows = selection_metadata["train_rows"]

expected_test_rows = selection_metadata["test_rows"]


expected_columns = expected_features + ["cardio"]

if train_df.shape[0] != expected_train_rows:
    raise ValueError(
        "Training-set row count does not match Feature Selection metadata.\n"
        f"Expected: {expected_train_rows:,}\n"
        f"Found:    {train_df.shape[0]:,}"
    )


if test_df.shape[0] != expected_test_rows:
    raise ValueError(
        "Test-set row count does not match Feature Selection metadata.\n"
        f"Expected: {expected_test_rows:,}\n"
        f"Found:    {test_df.shape[0]:,}"
    )


if list(train_df.columns) != expected_columns:
    raise ValueError(
        "Training-set columns do not match selected features.\n"
        f"Expected: {expected_columns}\n"
        f"Found:    {train_df.columns.tolist()}"
    )


if list(test_df.columns) != expected_columns:
    raise ValueError(
        "Test-set columns do not match selected features.\n"
        f"Expected: {expected_columns}\n"
        f"Found:    {test_df.columns.tolist()}"
    )


if train_df.isnull().sum().sum() != 0:
    raise ValueError("Missing values detected in training data.")


if test_df.isnull().sum().sum() != 0:
    raise ValueError("Missing values detected in test data.")


print("Soft Voting input-data validation passed.")

X_train = train_df[expected_features].copy()

y_train = train_df["cardio"].copy()


X_test = test_df[expected_features].copy()

y_test = test_df["cardio"].copy()


print(f"\nTraining Set: {len(X_train):,} rows")

print(f"Test Set (held out): {len(X_test):,} rows")

print(f"Predictors: {len(expected_features)}")

print("\nSelected predictors:")

for feature in expected_features:
    print(f"  - {feature}")

final_dt = joblib.load(DT_MODEL_FILE)

final_svm = joblib.load(SVM_MODEL_FILE)


with open(DT_SUMMARY_FILE, "r") as f:
    dt_summary = json.load(f)


with open(SVM_SUMMARY_FILE, "r") as f:
    svm_summary = json.load(f)


best_depth = dt_summary["best_depth"]

best_kernel = svm_summary["best_kernel"]


print("\nLoaded final fitted component models.")

print(f"Decision Tree max_depth: {best_depth}")

print(f"SVM kernel: {best_kernel}")

if dt_summary["selected_features"] != expected_features:
    raise ValueError(
        "Decision Tree feature list does not match "
        "the current Feature Selection output."
    )


if svm_summary["selected_features"] != expected_features:
    raise ValueError(
        "SVM feature list does not match the current Feature Selection output."
    )


if dt_summary["training_rows"] != expected_train_rows:
    raise ValueError("Decision Tree was trained on a different training-set version.")


if svm_summary["training_rows"] != expected_train_rows:
    raise ValueError("SVM was trained on a different training-set version.")


print("Component-model metadata validation passed.")

dt_cv_scores = np.load(DT_CV_SCORES_FILE)

svm_cv_scores = np.load(SVM_CV_SCORES_FILE)


if len(dt_cv_scores) != 5:
    raise ValueError("Decision Tree CV-score file does not contain 5 folds.")


if len(svm_cv_scores) != 5:
    raise ValueError("SVM CV-score file does not contain 5 folds.")


print("\nLoaded saved Decision Tree and SVM CV fold scores.")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


cv_splits = list(cv.split(X_train, y_train))


if len(cv_splits) != 5:
    raise ValueError("Expected exactly five CV folds.")

CV_INDICES_FILE = f"{ENSEMBLE_RESULTS_PATH}/cv_fold_indices.npz"


cv_index_dict = {}


for fold_number, (train_idx, val_idx) in enumerate(cv_splits, start=1):
    cv_index_dict[f"fold_{fold_number}_train"] = train_idx

    cv_index_dict[f"fold_{fold_number}_validation"] = val_idx


np.savez_compressed(CV_INDICES_FILE, **cv_index_dict)


print(f"\nExact CV fold indices saved to:\n{CV_INDICES_FILE}")

print("\n" + "=" * 75)

print("SOFT VOTING: 5-FOLD CROSS-VALIDATION")

print("=" * 75)


print("Using already-selected component hyperparameters:")

print(f"  Decision Tree max_depth = {best_depth}")

print(f"  SVM kernel = '{best_kernel}'")

print("\nNo hyperparameter tuning is performed in this block.")

print(
    "Fresh models are fitted only within each CV fold "
    "to obtain leakage-free ensemble validation scores.\n"
)


ensemble_cv_scores = []


for fold_number, (train_idx, val_idx) in enumerate(cv_splits, start=1):
    print(f"Processing Fold {fold_number}/5...")

    X_fold_train = X_train.iloc[train_idx]

    y_fold_train = y_train.iloc[train_idx]

    X_fold_val = X_train.iloc[val_idx]

    y_fold_val = y_train.iloc[val_idx]

    fold_dt = DecisionTreeClassifier(max_depth=best_depth, random_state=42)

    fold_dt.fit(X_fold_train, y_fold_train)

    dt_fold_prob = fold_dt.predict_proba(X_fold_val)[:, 1]

    fold_svm = Pipeline(
        [
            ("scaler", StandardScaler()),
            ("svm", SVC(kernel=best_kernel, probability=True, random_state=42)),
        ]
    )

    fold_svm.fit(X_fold_train, y_fold_train)

    svm_fold_prob = fold_svm.predict_proba(X_fold_val)[:, 1]

    # Equal-weight Soft Voting
    ensemble_fold_prob = (dt_fold_prob + svm_fold_prob) / 2.0

    ensemble_fold_pred = (ensemble_fold_prob >= 0.5).astype(int)

    fold_accuracy = accuracy_score(y_fold_val, ensemble_fold_pred)

    ensemble_cv_scores.append(fold_accuracy)

    print(f"  Fold {fold_number} Accuracy: {fold_accuracy * 100:.2f}%")


ensemble_cv_scores = np.asarray(ensemble_cv_scores, dtype=float)


ensemble_cv_mean = ensemble_cv_scores.mean()

ensemble_cv_std = ensemble_cv_scores.std()


print("\nSoft Voting CV result:")

print(f"Mean Accuracy: {ensemble_cv_mean * 100:.2f}%")

print(f"Standard Deviation: {ensemble_cv_std * 100:.2f}%")

ENSEMBLE_CV_SCORES_FILE = f"{ENSEMBLE_RESULTS_PATH}/soft_voting_cv_fold_scores.npy"


np.save(ENSEMBLE_CV_SCORES_FILE, ensemble_cv_scores)


print(f"\nSoft Voting CV scores saved to:\n{ENSEMBLE_CV_SCORES_FILE}")

cv_performance_df = pd.DataFrame(
    {
        "Fold": [1, 2, 3, 4, 5],
        "Decision_Tree": dt_cv_scores,
        "SVM": svm_cv_scores,
        "Soft_Voting_Ensemble": ensemble_cv_scores,
    }
)


CV_PERFORMANCE_FILE = f"{ENSEMBLE_RESULTS_PATH}/cv_performance_scores.csv"


cv_performance_df.to_csv(CV_PERFORMANCE_FILE, index=False)


print(f"\nPaired CV performance table saved to:\n{CV_PERFORMANCE_FILE}")


print("\nPaired CV scores:")

display(cv_performance_df.round(4))

ensemble_artifact = {
    "ensemble_type": "equal_weight_soft_voting",
    "decision_tree_model": final_dt,
    "svm_pipeline": final_svm,
    "weights": {"decision_tree": 0.5, "svm": 0.5},
    "decision_threshold": 0.5,
    "positive_class": 1,
    "selected_features": expected_features,
    "decision_tree_best_depth": best_depth,
    "svm_best_kernel": best_kernel,
}


ENSEMBLE_MODEL_FILE = f"{MODEL_PATH}/soft_voting_ensemble_final.joblib"


joblib.dump(ensemble_artifact, ENSEMBLE_MODEL_FILE)


print(f"\nSoft Voting ensemble artifact saved to:\n{ENSEMBLE_MODEL_FILE}")


def soft_voting_predict_proba(artifact, X):
    """
    Return two-class probabilities for the equal-weight
    pre-fitted Soft Voting ensemble.
    """

    feature_order = artifact["selected_features"]

    X_ordered = X[feature_order]

    dt_prob_positive = artifact["decision_tree_model"].predict_proba(X_ordered)[:, 1]

    svm_prob_positive = artifact["svm_pipeline"].predict_proba(X_ordered)[:, 1]

    dt_weight = artifact["weights"]["decision_tree"]

    svm_weight = artifact["weights"]["svm"]

    total_weight = dt_weight + svm_weight

    positive_probability = (
        (dt_weight * dt_prob_positive) + (svm_weight * svm_prob_positive)
    ) / total_weight

    return np.column_stack((1.0 - positive_probability, positive_probability))


def soft_voting_predict(artifact, X):

    probabilities = soft_voting_predict_proba(artifact, X)[:, 1]

    threshold = artifact["decision_threshold"]

    return (probabilities >= threshold).astype(int)


final_ensemble = joblib.load(ENSEMBLE_MODEL_FILE)


print("\nSaved Soft Voting artifact successfully reloaded.")

# FINAL TEST-SET EVALUATION
print("\n" + "=" * 75)

print("FINAL TEST SET PERFORMANCE — SOFT VOTING ENSEMBLE")

print("=" * 75)


ensemble_probabilities = soft_voting_predict_proba(final_ensemble, X_test)


y_prob_ensemble = ensemble_probabilities[:, 1]


y_pred_ensemble = soft_voting_predict(final_ensemble, X_test)

cm = confusion_matrix(y_test, y_pred_ensemble, labels=[0, 1])


tn, fp, fn, tp = cm.ravel()


specificity = tn / (tn + fp)


accuracy = accuracy_score(y_test, y_pred_ensemble)


precision = precision_score(y_test, y_pred_ensemble, zero_division=0)


recall = recall_score(y_test, y_pred_ensemble, zero_division=0)


f1 = f1_score(y_test, y_pred_ensemble, zero_division=0)


roc_auc = roc_auc_score(y_test, y_prob_ensemble)


average_precision = average_precision_score(y_test, y_prob_ensemble)


mcc = matthews_corrcoef(y_test, y_pred_ensemble)


kappa = cohen_kappa_score(y_test, y_pred_ensemble)


print(f"Accuracy:          {accuracy * 100:.2f}%")

print(f"Precision:         {precision * 100:.2f}%")

print(f"Recall:            {recall * 100:.2f}%")

print(f"Specificity:       {specificity * 100:.2f}%")

print(f"F1-Score:          {f1 * 100:.2f}%")

print(f"ROC-AUC:           {roc_auc:.4f}")

print(f"Average Precision: {average_precision:.4f}")

print(f"MCC:               {mcc:.4f}")

print(f"Cohen's Kappa:     {kappa:.4f}")


print("\nConfusion Matrix Values:")

print(f"TN = {tn:,}")

print(f"FP = {fp:,}")

print(f"FN = {fn:,}")

print(f"TP = {tp:,}")

ensemble_predictions_df = pd.DataFrame(
    {"y_true": y_test.to_numpy(), "y_pred": y_pred_ensemble, "y_prob": y_prob_ensemble}
)


ENSEMBLE_PREDICTIONS_FILE = f"{ENSEMBLE_RESULTS_PATH}/soft_voting_test_predictions.csv"


ensemble_predictions_df.to_csv(ENSEMBLE_PREDICTIONS_FILE, index=False)


print(f"\nSoft Voting test predictions saved to:\n{ENSEMBLE_PREDICTIONS_FILE}")

dt_test_metrics = dt_summary["test_metrics"]

svm_test_metrics = svm_summary["test_metrics"]


comparison_df = pd.DataFrame(
    [
        {
            "Model": "Decision Tree",
            "Accuracy": dt_test_metrics["accuracy"],
            "Precision": dt_test_metrics["precision"],
            "Recall": dt_test_metrics["recall"],
            "Specificity": dt_test_metrics["specificity"],
            "F1": dt_test_metrics["f1_score"],
            "ROC-AUC": dt_test_metrics["roc_auc"],
            "MCC": dt_test_metrics["mcc"],
            "Kappa": dt_test_metrics["cohen_kappa"],
        },
        {
            "Model": "Support Vector Machine",
            "Accuracy": svm_test_metrics["accuracy"],
            "Precision": svm_test_metrics["precision"],
            "Recall": svm_test_metrics["recall"],
            "Specificity": svm_test_metrics["specificity"],
            "F1": svm_test_metrics["f1_score"],
            "ROC-AUC": svm_test_metrics["roc_auc"],
            "MCC": svm_test_metrics["mcc"],
            "Kappa": svm_test_metrics["cohen_kappa"],
        },
        {
            "Model": "Soft Voting Ensemble",
            "Accuracy": accuracy,
            "Precision": precision,
            "Recall": recall,
            "Specificity": specificity,
            "F1": f1,
            "ROC-AUC": roc_auc,
            "MCC": mcc,
            "Kappa": kappa,
        },
    ]
)


MODEL_COMPARISON_FILE = f"{ENSEMBLE_RESULTS_PATH}/final_model_comparison.csv"


comparison_df.to_csv(MODEL_COMPARISON_FILE, index=False)


print("\nFinal model comparison:")

display(comparison_df.round(4))


print(f"Model comparison saved to:\n{MODEL_COMPARISON_FILE}")

ensemble_summary = {
    "model": "Equal-Weight Prefitted Soft Voting Ensemble",
    "components": {
        "decision_tree": {"model_file": DT_MODEL_FILE, "max_depth": int(best_depth)},
        "svm": {"model_file": SVM_MODEL_FILE, "kernel": best_kernel},
    },
    "selected_features": expected_features,
    "weights": {"decision_tree": 0.5, "svm": 0.5},
    "decision_threshold": 0.5,
    "cv_method": "StratifiedKFold",
    "cv_folds": 5,
    "cv_shuffle": True,
    "cv_random_state": 42,
    "ensemble_cv_fold_scores": ensemble_cv_scores.tolist(),
    "ensemble_cv_accuracy_mean": float(ensemble_cv_mean),
    "ensemble_cv_accuracy_std": float(ensemble_cv_std),
    "training_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
    "test_metrics": {
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "specificity": float(specificity),
        "f1_score": float(f1),
        "roc_auc": float(roc_auc),
        "average_precision": float(average_precision),
        "mcc": float(mcc),
        "cohen_kappa": float(kappa),
    },
    "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
    "ensemble_model_file": ENSEMBLE_MODEL_FILE,
    "ensemble_cv_scores_file": ENSEMBLE_CV_SCORES_FILE,
    "paired_cv_scores_file": CV_PERFORMANCE_FILE,
    "cv_fold_indices_file": CV_INDICES_FILE,
    "test_predictions_file": ENSEMBLE_PREDICTIONS_FILE,
    "model_comparison_file": MODEL_COMPARISON_FILE,
}


ENSEMBLE_SUMMARY_FILE = f"{ENSEMBLE_RESULTS_PATH}/soft_voting_summary.json"


with open(ENSEMBLE_SUMMARY_FILE, "w") as f:
    json.dump(ensemble_summary, f, indent=4)


print(f"\nSoft Voting summary saved to:\n{ENSEMBLE_SUMMARY_FILE}")

plt.figure(figsize=(6, 5))


sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    xticklabels=["No CVD (0)", "Has CVD (1)"],
    yticklabels=["No CVD (0)", "Has CVD (1)"],
)


plt.title("Soft Voting Ensemble Confusion Matrix (Test Set)")

plt.xlabel("Predicted Label")

plt.ylabel("True Label")

plt.tight_layout()


CONFUSION_MATRIX_FILE = f"{ENSEMBLE_RESULTS_PATH}/soft_voting_confusion_matrix.png"


plt.savefig(CONFUSION_MATRIX_FILE, dpi=300, bbox_inches="tight")

plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

fpr, tpr, _ = roc_curve(y_test, y_prob_ensemble)


axes[0].plot(
    fpr, tpr, color="darkorange", lw=2, label=(f"ROC curve (AUC = {roc_auc:.4f})")
)


axes[0].plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")


axes[0].set_xlim([0.0, 1.0])

axes[0].set_ylim([0.0, 1.05])

axes[0].set_xlabel("False Positive Rate (1 - Specificity)")

axes[0].set_ylabel("True Positive Rate (Recall/Sensitivity)")

axes[0].set_title("Receiver Operating Characteristic (ROC) Curve")

axes[0].legend(loc="lower right")

precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_prob_ensemble)


axes[1].plot(
    recall_curve,
    precision_curve,
    color="purple",
    lw=2,
    label=(f"PR curve (AP = {average_precision:.4f})"),
)


axes[1].set_xlabel("Recall (Sensitivity)")

axes[1].set_ylabel("Precision (Positive Predictive Value)")

axes[1].set_title("Precision-Recall Curve")

axes[1].legend(loc="lower left")


plt.tight_layout()


ROC_PR_FILE = f"{ENSEMBLE_RESULTS_PATH}/soft_voting_roc_pr_curves.png"


plt.savefig(ROC_PR_FILE, dpi=300, bbox_inches="tight")

plt.show()

ensemble_summary["figures"] = {
    "confusion_matrix": CONFUSION_MATRIX_FILE,
    "roc_pr_curves": ROC_PR_FILE,
}


with open(ENSEMBLE_SUMMARY_FILE, "w") as f:
    json.dump(ensemble_summary, f, indent=4)

print("\n" + "=" * 75)

print("SOFT VOTING ARTIFACTS SAVED")

print("=" * 75)


print(f"Ensemble artifact:\n  {ENSEMBLE_MODEL_FILE}")

print(f"\nEnsemble CV scores:\n  {ENSEMBLE_CV_SCORES_FILE}")

print(f"\nPaired DT/SVM/Ensemble CV scores:\n  {CV_PERFORMANCE_FILE}")

print(f"\nExact CV fold indices:\n  {CV_INDICES_FILE}")

print(f"\nEnsemble test predictions:\n  {ENSEMBLE_PREDICTIONS_FILE}")

print(f"\nModel comparison:\n  {MODEL_COMPARISON_FILE}")

print(f"\nEnsemble metadata:\n  {ENSEMBLE_SUMMARY_FILE}")

print(f"\nConfusion matrix figure:\n  {CONFUSION_MATRIX_FILE}")

print(f"\nROC / PR figure:\n  {ROC_PR_FILE}")


print("\nSoft Voting Ensemble evaluation complete.")

# Phase 3: Final Ensemble Selection, Explainability, and Statistical Analysis

## Phase 3 Setup and Artifact Validation

In [ ]:
import os
import json
import shutil
import joblib
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    cohen_kappa_score,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
)

from google.colab import drive

drive.mount("/content/drive")

sns.set_theme(style="whitegrid", palette="muted")

BASE_PATH = "/content/drive/MyDrive/COMP721 Project"

SELECTION_PATH = f"{BASE_PATH}/feature_selection"
MODEL_PATH = f"{BASE_PATH}/models"

DT_RESULTS_PATH = f"{BASE_PATH}/model_results/decision_tree"
SVM_RESULTS_PATH = f"{BASE_PATH}/model_results/svm"
SOFT_RESULTS_PATH = f"{BASE_PATH}/model_results/soft_voting"

PHASE3_PATH = f"{BASE_PATH}/phase3"
STACKING_RESULTS_PATH = f"{PHASE3_PATH}/stacking"
ENSEMBLE_SELECTION_PATH = f"{PHASE3_PATH}/ensemble_selection"
FINAL_RESULTS_PATH = f"{PHASE3_PATH}/final_results"
SHAP_RESULTS_PATH = f"{PHASE3_PATH}/shap"
STAT_RESULTS_PATH = f"{PHASE3_PATH}/statistical_analysis"
VALIDATION_PLOTS_PATH = f"{PHASE3_PATH}/validation_plots"

for path in [
    PHASE3_PATH,
    STACKING_RESULTS_PATH,
    ENSEMBLE_SELECTION_PATH,
    FINAL_RESULTS_PATH,
    SHAP_RESULTS_PATH,
    STAT_RESULTS_PATH,
    VALIDATION_PLOTS_PATH,
]:
    os.makedirs(path, exist_ok=True)

TRAIN_FILE = f"{SELECTION_PATH}/train_final.csv"
TEST_FILE = f"{SELECTION_PATH}/test_final.csv"
SELECTION_METADATA_FILE = f"{SELECTION_PATH}/feature_selection_summary.json"

DT_MODEL_FILE = f"{MODEL_PATH}/decision_tree_final.joblib"
SVM_MODEL_FILE = f"{MODEL_PATH}/svm_final_pipeline.joblib"
SOFT_MODEL_FILE = f"{MODEL_PATH}/soft_voting_ensemble_final.joblib"

DT_SUMMARY_FILE = f"{DT_RESULTS_PATH}/decision_tree_summary.json"
SVM_SUMMARY_FILE = f"{SVM_RESULTS_PATH}/svm_summary.json"
SOFT_SUMMARY_FILE = f"{SOFT_RESULTS_PATH}/soft_voting_summary.json"

DT_CV_FILE = f"{DT_RESULTS_PATH}/decision_tree_cv_fold_scores.npy"
SVM_CV_FILE = f"{SVM_RESULTS_PATH}/svm_cv_fold_scores.npy"
SOFT_CV_FILE = f"{SOFT_RESULTS_PATH}/soft_voting_cv_fold_scores.npy"

CV_INDICES_FILE = f"{SOFT_RESULTS_PATH}/cv_fold_indices.npz"
SOFT_TEST_PREDICTIONS_FILE = f"{SOFT_RESULTS_PATH}/soft_voting_test_predictions.csv"

required_files = [
    TRAIN_FILE,
    TEST_FILE,
    SELECTION_METADATA_FILE,
    DT_MODEL_FILE,
    SVM_MODEL_FILE,
    SOFT_MODEL_FILE,
    DT_SUMMARY_FILE,
    SVM_SUMMARY_FILE,
    SOFT_SUMMARY_FILE,
    DT_CV_FILE,
    SVM_CV_FILE,
    SOFT_CV_FILE,
    CV_INDICES_FILE,
    SOFT_TEST_PREDICTIONS_FILE,
]

missing_files = [path for path in required_files if not os.path.exists(path)]

if missing_files:
    raise FileNotFoundError(
        "Missing required Phase 2 artifacts:\n\n" + "\n".join(missing_files)
    )

train_df = pd.read_csv(TRAIN_FILE)
test_df = pd.read_csv(TEST_FILE)

with open(SELECTION_METADATA_FILE, "r") as f:
    selection_metadata = json.load(f)

with open(DT_SUMMARY_FILE, "r") as f:
    dt_summary = json.load(f)

with open(SVM_SUMMARY_FILE, "r") as f:
    svm_summary = json.load(f)

with open(SOFT_SUMMARY_FILE, "r") as f:
    soft_summary = json.load(f)


selected_features = selection_metadata["selected_features"]

X_train = train_df[selected_features].copy()
y_train = train_df["cardio"].copy()

X_test = test_df[selected_features].copy()
y_test = test_df["cardio"].copy()

final_dt = joblib.load(DT_MODEL_FILE)
final_svm = joblib.load(SVM_MODEL_FILE)
soft_voting_artifact = joblib.load(SOFT_MODEL_FILE)

dt_cv_scores = np.load(DT_CV_FILE)
svm_cv_scores = np.load(SVM_CV_FILE)
soft_cv_scores = np.load(SOFT_CV_FILE)

best_depth = int(dt_summary["best_depth"])
best_kernel = svm_summary["best_kernel"]

cv_indices = np.load(CV_INDICES_FILE)

outer_splits = []

for fold in range(1, 6):
    train_idx = cv_indices[f"fold_{fold}_train"]
    val_idx = cv_indices[f"fold_{fold}_validation"]
    outer_splits.append((train_idx, val_idx))

assert len(X_train) == selection_metadata["train_rows"]
assert len(X_test) == selection_metadata["test_rows"]
assert list(X_train.columns) == selected_features
assert len(dt_cv_scores) == 5
assert len(svm_cv_scores) == 5
assert len(soft_cv_scores) == 5
assert len(outer_splits) == 5

all_validation_indices = np.concatenate([val_idx for _, val_idx in outer_splits])

assert len(all_validation_indices) == len(X_train)
assert len(np.unique(all_validation_indices)) == len(X_train)

print("=" * 75)
print("PHASE 3 SETUP COMPLETE")
print("=" * 75)

print(f"Training rows:        {len(X_train):,}")
print(f"Held-out test rows:   {len(X_test):,}")
print(f"Selected predictors:  {len(selected_features)}")
print(f"Decision Tree depth:  {best_depth}")
print(f"SVM kernel:           {best_kernel}")

print("\nSelected features:")
for feature in selected_features:
    print(f"  - {feature}")

print("\nExisting aligned CV means:")
print(f"Decision Tree: {dt_cv_scores.mean() * 100:.2f}%")
print(f"SVM:           {svm_cv_scores.mean() * 100:.2f}%")
print(f"Soft Voting:   {soft_cv_scores.mean() * 100:.2f}%")

print("\nExact Phase 2 CV folds successfully restored.")

## Stacking Classifier

In [ ]:
def build_stacking_classifier():

    dt_base = DecisionTreeClassifier(max_depth=best_depth, random_state=42)

    svm_base = Pipeline(
        [
            ("scaler", StandardScaler()),
            ("svm", SVC(kernel=best_kernel, probability=False, random_state=42)),
        ]
    )

    meta_learner = Pipeline(
        [
            ("scaler", StandardScaler()),
            ("logistic_regression", LogisticRegression(max_iter=2000, random_state=42)),
        ]
    )

    inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    return StackingClassifier(
        estimators=[("decision_tree", dt_base), ("svm", svm_base)],
        final_estimator=meta_learner,
        cv=inner_cv,
        stack_method="auto",
        passthrough=False,
        n_jobs=-1,
    )


print("=" * 75)
print("STACKING CLASSIFIER: ALIGNED 5-FOLD CROSS-VALIDATION")
print("=" * 75)

print(f"Decision Tree max_depth = {best_depth}")
print(f"SVM kernel = '{best_kernel}'")
print("Meta-learner = Logistic Regression")
print("Test set is NOT used in this block.\n")


stacking_cv_scores = []
stacking_oof_records = []

start_time = time.time()

for fold_number, (train_idx, val_idx) in enumerate(outer_splits, start=1):
    print(f"Processing outer fold {fold_number}/5...")

    X_fold_train = X_train.iloc[train_idx]
    y_fold_train = y_train.iloc[train_idx]

    X_fold_val = X_train.iloc[val_idx]
    y_fold_val = y_train.iloc[val_idx]

    stacking_fold_model = build_stacking_classifier()

    stacking_fold_model.fit(X_fold_train, y_fold_train)

    fold_pred = stacking_fold_model.predict(X_fold_val)

    fold_prob = stacking_fold_model.predict_proba(X_fold_val)[:, 1]

    fold_accuracy = accuracy_score(y_fold_val, fold_pred)

    stacking_cv_scores.append(fold_accuracy)

    fold_records = pd.DataFrame(
        {
            "original_train_index": X_fold_val.index.to_numpy(),
            "fold": fold_number,
            "y_true": y_fold_val.to_numpy(),
            "y_pred": fold_pred,
            "y_prob": fold_prob,
        }
    )

    stacking_oof_records.append(fold_records)

    print(f"  Fold {fold_number} Accuracy: {fold_accuracy * 100:.2f}%")


stacking_cv_scores = np.asarray(stacking_cv_scores, dtype=float)

stacking_oof_df = pd.concat(stacking_oof_records, ignore_index=True)

elapsed_minutes = (time.time() - start_time) / 60


print("\n" + "=" * 75)
print("STACKING CROSS-VALIDATION RESULT")
print("=" * 75)

print(f"Mean Accuracy: {stacking_cv_scores.mean() * 100:.2f}%")

print(f"Standard Deviation: {stacking_cv_scores.std() * 100:.2f}%")

print(f"Elapsed Time: {elapsed_minutes:.2f} minutes")

# SAVE STACKING CV ARTIFACTS

STACKING_CV_FILE = f"{STACKING_RESULTS_PATH}/stacking_cv_fold_scores.npy"

STACKING_OOF_FILE = f"{STACKING_RESULTS_PATH}/stacking_oof_predictions.csv"

STACKING_CV_SUMMARY_FILE = f"{STACKING_RESULTS_PATH}/stacking_cv_summary.json"


np.save(STACKING_CV_FILE, stacking_cv_scores)

stacking_oof_df.to_csv(STACKING_OOF_FILE, index=False)


stacking_cv_summary = {
    "model": "StackingClassifier",
    "base_models": {"decision_tree_max_depth": best_depth, "svm_kernel": best_kernel},
    "meta_learner": "StandardScaler + LogisticRegression",
    "outer_cv_folds": 5,
    "inner_stacking_cv_folds": 5,
    "outer_cv_shuffle": True,
    "outer_cv_random_state": 42,
    "fold_scores": stacking_cv_scores.tolist(),
    "mean_accuracy": float(stacking_cv_scores.mean()),
    "std_accuracy": float(stacking_cv_scores.std()),
    "test_set_used": False,
}


with open(STACKING_CV_SUMMARY_FILE, "w") as f:
    json.dump(stacking_cv_summary, f, indent=4)


print("\nSaved:")
print(STACKING_CV_FILE)
print(STACKING_OOF_FILE)
print(STACKING_CV_SUMMARY_FILE)

## Soft Voting vs Stacking: Cross-Validation Ensemble Selection

In [ ]:
ensemble_cv_df = pd.DataFrame(
    {
        "Fold": np.arange(1, 6),
        "Soft_Voting": soft_cv_scores,
        "Stacking": stacking_cv_scores,
    }
)

ensemble_cv_df["Stacking_minus_Soft_Voting"] = (
    ensemble_cv_df["Stacking"] - ensemble_cv_df["Soft_Voting"]
)


soft_mean = soft_cv_scores.mean()
stacking_mean = stacking_cv_scores.mean()

soft_std = soft_cv_scores.std()
stacking_std = stacking_cv_scores.std()


if stacking_mean > soft_mean:
    selected_ensemble_type = "Stacking"
else:
    selected_ensemble_type = "Soft Voting"


print("=" * 75)
print("ENSEMBLE SELECTION — TRAINING CV ONLY")
print("=" * 75)

display(ensemble_cv_df.round(5))

print(f"\nSoft Voting: {soft_mean * 100:.2f}% (+/- {soft_std * 100:.2f}%)")

print(f"Stacking:    {stacking_mean * 100:.2f}% (+/- {stacking_std * 100:.2f}%)")

print(f"\nPRIMARY ENSEMBLE SELECTED: {selected_ensemble_type}")

print("\nSelection was based ONLY on aligned training cross-validation accuracy.")

ENSEMBLE_SELECTION_CSV = f"{ENSEMBLE_SELECTION_PATH}/soft_voting_vs_stacking_cv.csv"

ENSEMBLE_SELECTION_JSON = f"{ENSEMBLE_SELECTION_PATH}/ensemble_selection_summary.json"

ensemble_cv_df.to_csv(ENSEMBLE_SELECTION_CSV, index=False)


ensemble_selection_summary = {
    "selection_metric": "mean_5_fold_cv_accuracy",
    "test_set_used_for_selection": False,
    "soft_voting": {
        "fold_scores": soft_cv_scores.tolist(),
        "mean_accuracy": float(soft_mean),
        "std_accuracy": float(soft_std),
    },
    "stacking": {
        "fold_scores": stacking_cv_scores.tolist(),
        "mean_accuracy": float(stacking_mean),
        "std_accuracy": float(stacking_std),
    },
    "selected_ensemble": selected_ensemble_type,
    "tie_rule": "Soft Voting retained if mean CV accuracies are exactly equal.",
}


with open(ENSEMBLE_SELECTION_JSON, "w") as f:
    json.dump(ensemble_selection_summary, f, indent=4)

plt.figure(figsize=(9, 5))

plt.plot(
    ensemble_cv_df["Fold"],
    ensemble_cv_df["Soft_Voting"] * 100,
    marker="o",
    label="Soft Voting",
)

plt.plot(
    ensemble_cv_df["Fold"],
    ensemble_cv_df["Stacking"] * 100,
    marker="o",
    label="Stacking",
)

plt.xlabel("Cross-Validation Fold")
plt.ylabel("Accuracy (%)")
plt.title("Soft Voting vs Stacking: Aligned 5-Fold CV Accuracy")
plt.xticks([1, 2, 3, 4, 5])
plt.legend()
plt.tight_layout()

ENSEMBLE_CV_PLOT = f"{ENSEMBLE_SELECTION_PATH}/soft_voting_vs_stacking_cv.png"

plt.savefig(ENSEMBLE_CV_PLOT, dpi=300, bbox_inches="tight")

plt.show()


print("\nSelection artifacts saved.")

## Final Selected Ensemble and Held-Out Test Evaluation

In [ ]:
def soft_voting_predict_proba(artifact, X):

    features = artifact["selected_features"]

    X_ordered = X[features].copy()

    dt_prob = artifact["decision_tree_model"].predict_proba(X_ordered)[:, 1]

    svm_prob = artifact["svm_pipeline"].predict_proba(X_ordered)[:, 1]

    dt_weight = artifact["weights"]["decision_tree"]
    svm_weight = artifact["weights"]["svm"]

    positive_probability = (dt_weight * dt_prob + svm_weight * svm_prob) / (
        dt_weight + svm_weight
    )

    return np.column_stack([1.0 - positive_probability, positive_probability])


def calculate_binary_metrics(y_true, y_pred, y_prob):

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    tn, fp, fn, tp = cm.ravel()

    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan

    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "specificity": specificity,
        "f1_score": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "average_precision": average_precision_score(y_true, y_prob),
        "mcc": matthews_corrcoef(y_true, y_pred),
        "cohen_kappa": cohen_kappa_score(y_true, y_pred),
    }

    confusion = {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}

    return metrics, confusion, cm


if selected_ensemble_type == "Stacking":
    print("Stacking was selected from training CV.")

    print("Fitting final Stacking model on the complete training partition...")

    final_selected_ensemble = build_stacking_classifier()

    final_selected_ensemble.fit(X_train, y_train)

    SELECTED_ENSEMBLE_MODEL_FILE = f"{MODEL_PATH}/stacking_ensemble_final.joblib"

    joblib.dump(final_selected_ensemble, SELECTED_ENSEMBLE_MODEL_FILE)

    print("\nEvaluating Stacking ONCE on the held-out test set...")

    selected_y_pred = final_selected_ensemble.predict(X_test)

    selected_y_prob = final_selected_ensemble.predict_proba(X_test)[:, 1]


else:
    print("Soft Voting was selected from training CV.")

    print(
        "Reusing its previously saved held-out "
        "test predictions; no new test evaluation is performed."
    )

    final_selected_ensemble = soft_voting_artifact

    SELECTED_ENSEMBLE_MODEL_FILE = SOFT_MODEL_FILE

    saved_soft_predictions = pd.read_csv(SOFT_TEST_PREDICTIONS_FILE)

    if not np.array_equal(
        saved_soft_predictions["y_true"].to_numpy(), y_test.to_numpy()
    ):
        raise ValueError(
            "Saved Soft Voting predictions do not match "
            "the current held-out test labels."
        )

    selected_y_pred = saved_soft_predictions["y_pred"].to_numpy()

    selected_y_prob = saved_soft_predictions["y_prob"].to_numpy()

selected_metrics, selected_confusion, selected_cm = calculate_binary_metrics(
    y_test, selected_y_pred, selected_y_prob
)


print("\n" + "=" * 75)
print(f"FINAL SELECTED ENSEMBLE — {selected_ensemble_type.upper()}")
print("=" * 75)

print(f"Accuracy:          {selected_metrics['accuracy'] * 100:.2f}%")

print(f"Precision:         {selected_metrics['precision'] * 100:.2f}%")

print(f"Recall:            {selected_metrics['recall'] * 100:.2f}%")

print(f"Specificity:       {selected_metrics['specificity'] * 100:.2f}%")

print(f"F1-Score:          {selected_metrics['f1_score'] * 100:.2f}%")

print(f"ROC-AUC:           {selected_metrics['roc_auc']:.4f}")

print(f"Average Precision: {selected_metrics['average_precision']:.4f}")

print(f"MCC:               {selected_metrics['mcc']:.4f}")

print(f"Cohen's Kappa:     {selected_metrics['cohen_kappa']:.4f}")


print("\nConfusion Matrix:")
print(selected_cm)

SELECTED_TEST_PREDICTIONS_FILE = (
    f"{FINAL_RESULTS_PATH}/selected_ensemble_test_predictions.csv"
)

selected_prediction_df = pd.DataFrame(
    {"y_true": y_test.to_numpy(), "y_pred": selected_y_pred, "y_prob": selected_y_prob}
)

selected_prediction_df.to_csv(SELECTED_TEST_PREDICTIONS_FILE, index=False)

if selected_ensemble_type == "Stacking":
    selected_ensemble_cv_scores = stacking_cv_scores
else:
    selected_ensemble_cv_scores = soft_cv_scores

FINAL_ENSEMBLE_SUMMARY_FILE = f"{FINAL_RESULTS_PATH}/selected_ensemble_summary.json"


final_ensemble_summary = {
    "selected_ensemble": selected_ensemble_type,
    "selection_basis": "highest mean aligned 5-fold training CV accuracy",
    "test_set_used_for_selection": False,
    "selected_features": selected_features,
    "training_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
    "cv_fold_scores": selected_ensemble_cv_scores.tolist(),
    "cv_accuracy_mean": float(selected_ensemble_cv_scores.mean()),
    "cv_accuracy_std": float(selected_ensemble_cv_scores.std()),
    "test_metrics": {key: float(value) for key, value in selected_metrics.items()},
    "confusion_matrix": selected_confusion,
    "model_file": SELECTED_ENSEMBLE_MODEL_FILE,
    "test_predictions_file": SELECTED_TEST_PREDICTIONS_FILE,
}


with open(FINAL_ENSEMBLE_SUMMARY_FILE, "w") as f:
    json.dump(final_ensemble_summary, f, indent=4)

plt.figure(figsize=(6, 5))

sns.heatmap(
    selected_cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    xticklabels=["No CVD (0)", "Has CVD (1)"],
    yticklabels=["No CVD (0)", "Has CVD (1)"],
)

plt.title(f"{selected_ensemble_type} Confusion Matrix (Held-Out Test Set)")

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()

SELECTED_CM_FILE = f"{FINAL_RESULTS_PATH}/selected_ensemble_confusion_matrix.png"

plt.savefig(SELECTED_CM_FILE, dpi=300, bbox_inches="tight")

plt.show()

fpr, tpr, _ = roc_curve(y_test, selected_y_prob)

plt.figure(figsize=(7, 6))

plt.plot(
    fpr,
    tpr,
    lw=2,
    label=(f"{selected_ensemble_type} (AUC = {selected_metrics['roc_auc']:.4f})"),
)

plt.plot([0, 1], [0, 1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — Selected Ensemble")
plt.legend(loc="lower right")
plt.tight_layout()

SELECTED_ROC_FILE = f"{FINAL_RESULTS_PATH}/selected_ensemble_roc_curve.png"

plt.savefig(SELECTED_ROC_FILE, dpi=300, bbox_inches="tight")

plt.show()

precision_curve, recall_curve, _ = precision_recall_curve(y_test, selected_y_prob)

plt.figure(figsize=(7, 6))

plt.plot(
    recall_curve,
    precision_curve,
    lw=2,
    label=(
        f"{selected_ensemble_type} (AP = {selected_metrics['average_precision']:.4f})"
    ),
)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve — Selected Ensemble")

plt.legend(loc="lower left")
plt.tight_layout()

SELECTED_PR_FILE = f"{FINAL_RESULTS_PATH}/selected_ensemble_precision_recall_curve.png"

plt.savefig(SELECTED_PR_FILE, dpi=300, bbox_inches="tight")

plt.show()

print("\nFinal selected ensemble artifacts saved.")

## Hyperparameter Validation Performance

In [ ]:
with open(DT_SUMMARY_FILE, "r") as f:
    dt_summary_plot = json.load(f)

with open(SVM_SUMMARY_FILE, "r") as f:
    svm_summary_plot = json.load(f)

depths = [int(depth) for depth in dt_summary_plot["tested_depths"]]

dt_means = [
    dt_summary_plot["cv_results"][str(depth)]["mean_accuracy"] * 100 for depth in depths
]

dt_stds = [
    dt_summary_plot["cv_results"][str(depth)]["std_accuracy"] * 100 for depth in depths
]


plt.figure(figsize=(8, 5))

plt.errorbar(depths, dt_means, yerr=dt_stds, marker="o", capsize=5)

plt.xlabel("Maximum Tree Depth")
plt.ylabel("Mean 5-Fold CV Accuracy (%)")
plt.title("Decision Tree Hyperparameter Validation Performance")

plt.xticks(depths)
plt.tight_layout()

DT_VALIDATION_PLOT = f"{VALIDATION_PLOTS_PATH}/decision_tree_validation_performance.png"

plt.savefig(DT_VALIDATION_PLOT, dpi=300, bbox_inches="tight")

plt.show()

kernels = svm_summary_plot["tested_kernels"]

svm_means = [
    svm_summary_plot["cv_results"][kernel]["mean_accuracy"] * 100 for kernel in kernels
]

svm_stds = [
    svm_summary_plot["cv_results"][kernel]["std_accuracy"] * 100 for kernel in kernels
]


plt.figure(figsize=(8, 5))

positions = np.arange(len(kernels))

plt.bar(positions, svm_means, yerr=svm_stds, capsize=5)

plt.xticks(positions, [kernel.upper() for kernel in kernels])

plt.xlabel("SVM Kernel")
plt.ylabel("Mean 5-Fold CV Accuracy (%)")
plt.title("SVM Hyperparameter Validation Performance")

plt.tight_layout()

SVM_VALIDATION_PLOT = f"{VALIDATION_PLOTS_PATH}/svm_validation_performance.png"

plt.savefig(SVM_VALIDATION_PLOT, dpi=300, bbox_inches="tight")

plt.show()


print("Required hyperparameter validation plots saved.")

## Final Model Performance Comparison

In [ ]:
dt_metrics = dt_summary["test_metrics"]
svm_metrics = svm_summary["test_metrics"]


final_comparison_df = pd.DataFrame(
    [
        {
            "Model": "Decision Tree",
            "Accuracy": dt_metrics["accuracy"],
            "Precision": dt_metrics["precision"],
            "Recall": dt_metrics["recall"],
            "Specificity": dt_metrics["specificity"],
            "F1": dt_metrics["f1_score"],
            "ROC-AUC": dt_metrics["roc_auc"],
            "Average Precision": dt_metrics["average_precision"],
            "MCC": dt_metrics["mcc"],
            "Kappa": dt_metrics["cohen_kappa"],
        },
        {
            "Model": "Support Vector Machine",
            "Accuracy": svm_metrics["accuracy"],
            "Precision": svm_metrics["precision"],
            "Recall": svm_metrics["recall"],
            "Specificity": svm_metrics["specificity"],
            "F1": svm_metrics["f1_score"],
            "ROC-AUC": svm_metrics["roc_auc"],
            "Average Precision": svm_metrics["average_precision"],
            "MCC": svm_metrics["mcc"],
            "Kappa": svm_metrics["cohen_kappa"],
        },
        {
            "Model": selected_ensemble_type,
            "Accuracy": selected_metrics["accuracy"],
            "Precision": selected_metrics["precision"],
            "Recall": selected_metrics["recall"],
            "Specificity": selected_metrics["specificity"],
            "F1": selected_metrics["f1_score"],
            "ROC-AUC": selected_metrics["roc_auc"],
            "Average Precision": selected_metrics["average_precision"],
            "MCC": selected_metrics["mcc"],
            "Kappa": selected_metrics["cohen_kappa"],
        },
    ]
)


display(final_comparison_df.round(4))


FINAL_COMPARISON_FILE = f"{FINAL_RESULTS_PATH}/final_model_comparison.csv"

final_comparison_df.to_csv(FINAL_COMPARISON_FILE, index=False)

dt_predictions = pd.read_csv(dt_summary["test_predictions_file"])

svm_predictions = pd.read_csv(svm_summary["test_predictions_file"])

plt.figure(figsize=(8, 6))

for model_name, prediction_df in [
    ("Decision Tree", dt_predictions),
    ("SVM", svm_predictions),
    (selected_ensemble_type, selected_prediction_df),
]:
    fpr_model, tpr_model, _ = roc_curve(
        prediction_df["y_true"], prediction_df["y_prob"]
    )

    auc_model = roc_auc_score(prediction_df["y_true"], prediction_df["y_prob"])

    plt.plot(fpr_model, tpr_model, lw=2, label=(f"{model_name} (AUC={auc_model:.4f})"))


plt.plot([0, 1], [0, 1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Comparison of Final Models")
plt.legend(loc="lower right")
plt.tight_layout()

COMBINED_ROC_FILE = f"{FINAL_RESULTS_PATH}/combined_roc_curves.png"

plt.savefig(COMBINED_ROC_FILE, dpi=300, bbox_inches="tight")

plt.show()

plt.figure(figsize=(8, 6))

for model_name, prediction_df in [
    ("Decision Tree", dt_predictions),
    ("SVM", svm_predictions),
    (selected_ensemble_type, selected_prediction_df),
]:
    precision_model, recall_model, _ = precision_recall_curve(
        prediction_df["y_true"], prediction_df["y_prob"]
    )

    ap_model = average_precision_score(prediction_df["y_true"], prediction_df["y_prob"])

    plt.plot(
        recall_model, precision_model, lw=2, label=(f"{model_name} (AP={ap_model:.4f})")
    )


plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Comparison of Final Models")

plt.legend(loc="lower left")
plt.tight_layout()

COMBINED_PR_FILE = f"{FINAL_RESULTS_PATH}/combined_precision_recall_curves.png"

plt.savefig(COMBINED_PR_FILE, dpi=300, bbox_inches="tight")

plt.show()


print(f"Final comparison saved to:\n{FINAL_COMPARISON_FILE}")

# Explainable AI (SHAP)

## Decision Tree SHAP Analysis

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive


try:
    import shap
except ImportError:
    !pip install -q shap
    import shap

drive.mount("/content/drive")

BASE_PATH = "/content/drive/MyDrive/COMP721 Project"

SELECTION_PATH = f"{BASE_PATH}/feature_selection"

MODEL_PATH = f"{BASE_PATH}/models"

SHAP_RESULTS_PATH = f"{BASE_PATH}/phase3/shap"

os.makedirs(SHAP_RESULTS_PATH, exist_ok=True)


TRAIN_FILE = f"{SELECTION_PATH}/train_final.csv"

TEST_FILE = f"{SELECTION_PATH}/test_final.csv"

DT_MODEL_FILE = f"{MODEL_PATH}/decision_tree_final.joblib"

DT_SHAP_FILE = f"{SHAP_RESULTS_PATH}/decision_tree_shap_values.npz"

train_df = pd.read_csv(TRAIN_FILE)

test_df = pd.read_csv(TEST_FILE)


selected_features = [col for col in train_df.columns if col != "cardio"]


X_train = train_df[selected_features].copy()

X_test = test_df[selected_features].copy()

y_test = test_df["cardio"].copy()

decision_tree = joblib.load(DT_MODEL_FILE)

BACKGROUND_SIZE = 50
EXPLANATION_SIZE = 200


background_df = X_train.sample(
    n=min(BACKGROUND_SIZE, len(X_train)), random_state=42
).reset_index(drop=True)


explanation_indices = X_test.sample(
    n=min(EXPLANATION_SIZE, len(X_test)), random_state=42
).index


X_explain = X_test.loc[explanation_indices].copy()


def decision_tree_probability(X):

    if not isinstance(X, pd.DataFrame):
        X = pd.DataFrame(X, columns=selected_features)

    else:
        X = X[selected_features]

    return decision_tree.predict_proba(X)[:, 1]


if os.path.exists(DT_SHAP_FILE):
    print("Loading previously saved Decision Tree SHAP values...")

    saved = np.load(DT_SHAP_FILE, allow_pickle=True)

    explanation = shap.Explanation(
        values=saved["values"],
        base_values=saved["base_values"],
        data=saved["data"],
        feature_names=selected_features,
    )


else:
    print("Computing Decision Tree SHAP values...")

    masker = shap.maskers.Independent(background_df, max_samples=BACKGROUND_SIZE)

    explainer = shap.Explainer(
        decision_tree_probability,
        masker,
        algorithm="permutation",
        feature_names=selected_features,
        seed=42,
    )

    NUM_FEATURES = len(selected_features)

    MAX_EVALS = 5 * (2 * NUM_FEATURES + 1)

    explanation = explainer(X_explain, max_evals=MAX_EVALS, batch_size="auto")

    np.savez_compressed(
        DT_SHAP_FILE,
        values=explanation.values,
        base_values=explanation.base_values,
        data=explanation.data,
        test_indices=explanation_indices.to_numpy(),
        feature_names=np.array(selected_features),
    )

    print(f"SHAP values saved to:\n{DT_SHAP_FILE}")

plt.figure()

shap.summary_plot(
    explanation.values, X_explain, feature_names=selected_features, show=False
)

plt.title("SHAP Summary Plot — Decision Tree")

plt.tight_layout()


DT_SUMMARY_PLOT = f"{SHAP_RESULTS_PATH}/decision_tree_shap_summary.png"


plt.savefig(DT_SUMMARY_PLOT, dpi=300, bbox_inches="tight")

plt.show()

plt.figure()

shap.summary_plot(
    explanation.values,
    X_explain,
    feature_names=selected_features,
    plot_type="bar",
    show=False,
)

plt.title("SHAP Global Feature Importance — Decision Tree")

plt.tight_layout()


DT_BAR_PLOT = f"{SHAP_RESULTS_PATH}/decision_tree_shap_bar.png"


plt.savefig(DT_BAR_PLOT, dpi=300, bbox_inches="tight")

plt.show()

importance_df = pd.DataFrame(
    {
        "Feature": selected_features,
        "Mean_Absolute_SHAP": np.mean(np.abs(explanation.values), axis=0),
    }
)


importance_df = importance_df.sort_values(
    "Mean_Absolute_SHAP", ascending=False
).reset_index(drop=True)


print("\nDecision Tree SHAP Feature Importance:")

display(importance_df.round(6))


print("\nDecision Tree SHAP analysis complete.")

## Support Vector Machine SHAP Analysis

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive


try:
    import shap
except ImportError:
    !pip install -q shap
    import shap

drive.mount("/content/drive")

BASE_PATH = "/content/drive/MyDrive/COMP721 Project"

SELECTION_PATH = f"{BASE_PATH}/feature_selection"

MODEL_PATH = f"{BASE_PATH}/models"

SHAP_RESULTS_PATH = f"{BASE_PATH}/phase3/shap"

os.makedirs(SHAP_RESULTS_PATH, exist_ok=True)


TRAIN_FILE = f"{SELECTION_PATH}/train_final.csv"

TEST_FILE = f"{SELECTION_PATH}/test_final.csv"

SVM_MODEL_FILE = f"{MODEL_PATH}/svm_final_pipeline.joblib"

SVM_SHAP_FILE = f"{SHAP_RESULTS_PATH}/svm_shap_values.npz"

train_df = pd.read_csv(TRAIN_FILE)

test_df = pd.read_csv(TEST_FILE)


selected_features = [col for col in train_df.columns if col != "cardio"]


X_train = train_df[selected_features].copy()

X_test = test_df[selected_features].copy()

svm_pipeline = joblib.load(SVM_MODEL_FILE)

BACKGROUND_SIZE = 50
EXPLANATION_SIZE = 200


background_df = X_train.sample(
    n=min(BACKGROUND_SIZE, len(X_train)), random_state=42
).reset_index(drop=True)


explanation_indices = X_test.sample(
    n=min(EXPLANATION_SIZE, len(X_test)), random_state=42
).index


X_explain = X_test.loc[explanation_indices].copy()


def svm_probability(X):

    if not isinstance(X, pd.DataFrame):
        X = pd.DataFrame(X, columns=selected_features)

    else:
        X = X[selected_features]

    return svm_pipeline.predict_proba(X)[:, 1]


if os.path.exists(SVM_SHAP_FILE):
    print("Loading previously saved SVM SHAP values...")

    saved = np.load(SVM_SHAP_FILE, allow_pickle=True)

    explanation = shap.Explanation(
        values=saved["values"],
        base_values=saved["base_values"],
        data=saved["data"],
        feature_names=selected_features,
    )


else:
    print("Computing SVM SHAP values...")

    masker = shap.maskers.Independent(background_df, max_samples=BACKGROUND_SIZE)

    explainer = shap.Explainer(
        svm_probability,
        masker,
        algorithm="permutation",
        feature_names=selected_features,
        seed=42,
    )

    NUM_FEATURES = len(selected_features)

    MAX_EVALS = 5 * (2 * NUM_FEATURES + 1)

    explanation = explainer(X_explain, max_evals=MAX_EVALS, batch_size="auto")

    np.savez_compressed(
        SVM_SHAP_FILE,
        values=explanation.values,
        base_values=explanation.base_values,
        data=explanation.data,
        test_indices=explanation_indices.to_numpy(),
        feature_names=np.array(selected_features),
    )

    print(f"SHAP values saved to:\n{SVM_SHAP_FILE}")

plt.figure()

shap.summary_plot(
    explanation.values, X_explain, feature_names=selected_features, show=False
)

plt.title("SHAP Summary Plot — Support Vector Machine")

plt.tight_layout()


SVM_SUMMARY_PLOT = f"{SHAP_RESULTS_PATH}/svm_shap_summary.png"


plt.savefig(SVM_SUMMARY_PLOT, dpi=300, bbox_inches="tight")

plt.show()

plt.figure()

shap.summary_plot(
    explanation.values,
    X_explain,
    feature_names=selected_features,
    plot_type="bar",
    show=False,
)

plt.title("SHAP Global Feature Importance — Support Vector Machine")

plt.tight_layout()


SVM_BAR_PLOT = f"{SHAP_RESULTS_PATH}/svm_shap_bar.png"


plt.savefig(SVM_BAR_PLOT, dpi=300, bbox_inches="tight")

plt.show()

importance_df = pd.DataFrame(
    {
        "Feature": selected_features,
        "Mean_Absolute_SHAP": np.mean(np.abs(explanation.values), axis=0),
    }
)


importance_df = importance_df.sort_values(
    "Mean_Absolute_SHAP", ascending=False
).reset_index(drop=True)


print("\nSVM SHAP Feature Importance:")

display(importance_df.round(6))


print("\nSVM SHAP analysis complete.")

## Soft Voting Ensemble SHAP Analysis

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive


try:
    import shap
except ImportError:
    !pip install -q shap
    import shap

drive.mount("/content/drive")

BASE_PATH = "/content/drive/MyDrive/COMP721 Project"

SELECTION_PATH = f"{BASE_PATH}/feature_selection"

MODEL_PATH = f"{BASE_PATH}/models"

SHAP_RESULTS_PATH = f"{BASE_PATH}/phase3/shap"

os.makedirs(SHAP_RESULTS_PATH, exist_ok=True)


TRAIN_FILE = f"{SELECTION_PATH}/train_final.csv"

TEST_FILE = f"{SELECTION_PATH}/test_final.csv"

SOFT_MODEL_FILE = f"{MODEL_PATH}/soft_voting_ensemble_final.joblib"

SOFT_SHAP_FILE = f"{SHAP_RESULTS_PATH}/soft_voting_shap_values.npz"

train_df = pd.read_csv(TRAIN_FILE)

test_df = pd.read_csv(TEST_FILE)


selected_features = [col for col in train_df.columns if col != "cardio"]


X_train = train_df[selected_features].copy()

X_test = test_df[selected_features].copy()

soft_artifact = joblib.load(SOFT_MODEL_FILE)


decision_tree = soft_artifact["decision_tree_model"]

svm_pipeline = soft_artifact["svm_pipeline"]

dt_weight = soft_artifact["weights"]["decision_tree"]

svm_weight = soft_artifact["weights"]["svm"]

BACKGROUND_SIZE = 50
EXPLANATION_SIZE = 200


background_df = X_train.sample(
    n=min(BACKGROUND_SIZE, len(X_train)), random_state=42
).reset_index(drop=True)


explanation_indices = X_test.sample(
    n=min(EXPLANATION_SIZE, len(X_test)), random_state=42
).index


X_explain = X_test.loc[explanation_indices].copy()


def soft_voting_probability(X):

    if not isinstance(X, pd.DataFrame):
        X = pd.DataFrame(X, columns=selected_features)

    else:
        X = X[selected_features]

    dt_probability = decision_tree.predict_proba(X)[:, 1]

    svm_probability = svm_pipeline.predict_proba(X)[:, 1]

    probability = ((dt_weight * dt_probability) + (svm_weight * svm_probability)) / (
        dt_weight + svm_weight
    )

    return probability


if os.path.exists(SOFT_SHAP_FILE):
    print("Loading previously saved Soft Voting SHAP values...")

    saved = np.load(SOFT_SHAP_FILE, allow_pickle=True)

    explanation = shap.Explanation(
        values=saved["values"],
        base_values=saved["base_values"],
        data=saved["data"],
        feature_names=selected_features,
    )


else:
    print("Computing Soft Voting SHAP values...")

    masker = shap.maskers.Independent(background_df, max_samples=BACKGROUND_SIZE)

    explainer = shap.Explainer(
        soft_voting_probability,
        masker,
        algorithm="permutation",
        feature_names=selected_features,
        seed=42,
    )

    NUM_FEATURES = len(selected_features)

    MAX_EVALS = 5 * (2 * NUM_FEATURES + 1)

    explanation = explainer(X_explain, max_evals=MAX_EVALS, batch_size="auto")

    np.savez_compressed(
        SOFT_SHAP_FILE,
        values=explanation.values,
        base_values=explanation.base_values,
        data=explanation.data,
        test_indices=explanation_indices.to_numpy(),
        feature_names=np.array(selected_features),
    )

    print(f"SHAP values saved to:\n{SOFT_SHAP_FILE}")

plt.figure()

shap.summary_plot(
    explanation.values, X_explain, feature_names=selected_features, show=False
)

plt.title("SHAP Summary Plot — Soft Voting Ensemble")

plt.tight_layout()


SOFT_SUMMARY_PLOT = f"{SHAP_RESULTS_PATH}/soft_voting_shap_summary.png"


plt.savefig(SOFT_SUMMARY_PLOT, dpi=300, bbox_inches="tight")

plt.show()

plt.figure()

shap.summary_plot(
    explanation.values,
    X_explain,
    feature_names=selected_features,
    plot_type="bar",
    show=False,
)

plt.title("SHAP Global Feature Importance — Soft Voting Ensemble")

plt.tight_layout()


SOFT_BAR_PLOT = f"{SHAP_RESULTS_PATH}/soft_voting_shap_bar.png"


plt.savefig(SOFT_BAR_PLOT, dpi=300, bbox_inches="tight")

plt.show()

importance_df = pd.DataFrame(
    {
        "Feature": selected_features,
        "Mean_Absolute_SHAP": np.mean(np.abs(explanation.values), axis=0),
    }
)


importance_df = importance_df.sort_values(
    "Mean_Absolute_SHAP", ascending=False
).reset_index(drop=True)


print("\nSoft Voting SHAP Feature Importance:")

display(importance_df.round(6))


print("\nSoft Voting SHAP analysis complete.")

## SHAP Feature Importance Comparison

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import drive


try:
    import shap
except ImportError:
    !pip install -q shap
    import shap

drive.mount("/content/drive")

BASE_PATH = "/content/drive/MyDrive/COMP721 Project"

SELECTION_PATH = f"{BASE_PATH}/feature_selection"

MODEL_PATH = f"{BASE_PATH}/models"

SHAP_RESULTS_PATH = f"{BASE_PATH}/phase3/shap"

os.makedirs(SHAP_RESULTS_PATH, exist_ok=True)


TRAIN_FILE = f"{SELECTION_PATH}/train_final.csv"

TEST_FILE = f"{SELECTION_PATH}/test_final.csv"

DT_MODEL_FILE = f"{MODEL_PATH}/decision_tree_final.joblib"

SVM_MODEL_FILE = f"{MODEL_PATH}/svm_final_pipeline.joblib"

SOFT_MODEL_FILE = f"{MODEL_PATH}/soft_voting_ensemble_final.joblib"


DT_SHAP_FILE = f"{SHAP_RESULTS_PATH}/decision_tree_shap_values.npz"

SVM_SHAP_FILE = f"{SHAP_RESULTS_PATH}/svm_shap_values.npz"

SOFT_SHAP_FILE = f"{SHAP_RESULTS_PATH}/soft_voting_shap_values.npz"

train_df = pd.read_csv(TRAIN_FILE)

test_df = pd.read_csv(TEST_FILE)


selected_features = [col for col in train_df.columns if col != "cardio"]


X_train = train_df[selected_features].copy()

X_test = test_df[selected_features].copy()

decision_tree = joblib.load(DT_MODEL_FILE)

svm_pipeline = joblib.load(SVM_MODEL_FILE)

soft_artifact = joblib.load(SOFT_MODEL_FILE)


soft_dt = soft_artifact["decision_tree_model"]

soft_svm = soft_artifact["svm_pipeline"]

dt_weight = soft_artifact["weights"]["decision_tree"]

svm_weight = soft_artifact["weights"]["svm"]

BACKGROUND_SIZE = 50
EXPLANATION_SIZE = 200


background_df = X_train.sample(
    n=min(BACKGROUND_SIZE, len(X_train)), random_state=42
).reset_index(drop=True)


explanation_indices = X_test.sample(
    n=min(EXPLANATION_SIZE, len(X_test)), random_state=42
).index


X_explain = X_test.loc[explanation_indices].copy()


NUM_FEATURES = len(selected_features)

MAX_EVALS = 5 * (2 * NUM_FEATURES + 1)


def to_frame(X):

    if isinstance(X, pd.DataFrame):
        return X[selected_features]

    return pd.DataFrame(X, columns=selected_features)


def dt_probability(X):

    X = to_frame(X)

    return decision_tree.predict_proba(X)[:, 1]


def svm_probability(X):

    X = to_frame(X)

    return svm_pipeline.predict_proba(X)[:, 1]


def soft_probability(X):

    X = to_frame(X)

    dt_prob = soft_dt.predict_proba(X)[:, 1]

    svm_prob = soft_svm.predict_proba(X)[:, 1]

    return (dt_weight * dt_prob + svm_weight * svm_prob) / (dt_weight + svm_weight)


def load_or_compute_shap(model_name, prediction_function, file_path):

    if os.path.exists(file_path):
        print(f"Loading saved SHAP values for {model_name}...")

        saved = np.load(file_path, allow_pickle=True)

        return shap.Explanation(
            values=saved["values"],
            base_values=saved["base_values"],
            data=saved["data"],
            feature_names=selected_features,
        )

    print(f"Computing SHAP values for {model_name}...")

    masker = shap.maskers.Independent(background_df, max_samples=BACKGROUND_SIZE)

    explainer = shap.Explainer(
        prediction_function,
        masker,
        algorithm="permutation",
        feature_names=selected_features,
        seed=42,
    )

    explanation = explainer(X_explain, max_evals=MAX_EVALS, batch_size="auto")

    np.savez_compressed(
        file_path,
        values=explanation.values,
        base_values=explanation.base_values,
        data=explanation.data,
        test_indices=explanation_indices.to_numpy(),
        feature_names=np.array(selected_features),
    )

    return explanation


dt_shap = load_or_compute_shap("Decision Tree", dt_probability, DT_SHAP_FILE)

svm_shap = load_or_compute_shap("SVM", svm_probability, SVM_SHAP_FILE)

soft_shap = load_or_compute_shap("Soft Voting", soft_probability, SOFT_SHAP_FILE)

comparison_df = pd.DataFrame(
    {
        "Feature": selected_features,
        "Decision Tree": np.mean(np.abs(dt_shap.values), axis=0),
        "SVM": np.mean(np.abs(svm_shap.values), axis=0),
        "Soft Voting": np.mean(np.abs(soft_shap.values), axis=0),
    }
)


for model_name in ["Decision Tree", "SVM", "Soft Voting"]:
    comparison_df[f"{model_name} Rank"] = (
        comparison_df[model_name].rank(ascending=False, method="min").astype(int)
    )


print("=" * 75)

print("SHAP FEATURE IMPORTANCE COMPARISON")

print("=" * 75)


display(comparison_df.round(6))


SHAP_COMPARISON_CSV = f"{SHAP_RESULTS_PATH}/shap_feature_importance_comparison.csv"


comparison_df.to_csv(SHAP_COMPARISON_CSV, index=False)

plot_df = comparison_df[["Feature", "Decision Tree", "SVM", "Soft Voting"]].melt(
    id_vars="Feature", var_name="Model", value_name="Mean_Absolute_SHAP"
)


plt.figure(figsize=(10, 6))


sns.barplot(data=plot_df, x="Mean_Absolute_SHAP", y="Feature", hue="Model")


plt.xlabel("Mean |SHAP Value|")

plt.ylabel("Feature")

plt.title("SHAP Feature Importance Comparison Across Models")

plt.tight_layout()


SHAP_COMPARISON_PLOT = f"{SHAP_RESULTS_PATH}/shap_importance_model_comparison.png"


plt.savefig(SHAP_COMPARISON_PLOT, dpi=300, bbox_inches="tight")

plt.show()


print("\nSHAP comparison complete.")

## SHAP Dependence Plot

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive


try:
    import shap
except ImportError:
    !pip install -q shap
    import shap


drive.mount("/content/drive")


BASE_PATH = "/content/drive/MyDrive/COMP721 Project"

SELECTION_PATH = f"{BASE_PATH}/feature_selection"

MODEL_PATH = f"{BASE_PATH}/models"

SHAP_RESULTS_PATH = f"{BASE_PATH}/phase3/shap"

os.makedirs(SHAP_RESULTS_PATH, exist_ok=True)


TRAIN_FILE = f"{SELECTION_PATH}/train_final.csv"

TEST_FILE = f"{SELECTION_PATH}/test_final.csv"

SOFT_MODEL_FILE = f"{MODEL_PATH}/soft_voting_ensemble_final.joblib"

SOFT_SHAP_FILE = f"{SHAP_RESULTS_PATH}/soft_voting_shap_values.npz"


train_df = pd.read_csv(TRAIN_FILE)

test_df = pd.read_csv(TEST_FILE)


selected_features = [col for col in train_df.columns if col != "cardio"]


X_train = train_df[selected_features]

X_test = test_df[selected_features]


BACKGROUND_SIZE = 50
EXPLANATION_SIZE = 200


background_df = X_train.sample(
    n=min(BACKGROUND_SIZE, len(X_train)), random_state=42
).reset_index(drop=True)


explanation_indices = X_test.sample(
    n=min(EXPLANATION_SIZE, len(X_test)), random_state=42
).index


X_explain = X_test.loc[explanation_indices].copy()


if os.path.exists(SOFT_SHAP_FILE):
    print("Loading saved Soft Voting SHAP values...")

    saved = np.load(SOFT_SHAP_FILE, allow_pickle=True)

    soft_shap = shap.Explanation(
        values=saved["values"],
        base_values=saved["base_values"],
        data=saved["data"],
        feature_names=selected_features,
    )


else:
    print("Soft Voting SHAP values not found. Computing them now...")

    artifact = joblib.load(SOFT_MODEL_FILE)

    dt_model = artifact["decision_tree_model"]

    svm_model = artifact["svm_pipeline"]

    dt_weight = artifact["weights"]["decision_tree"]

    svm_weight = artifact["weights"]["svm"]

    def probability(X):

        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X, columns=selected_features)

        dt_prob = dt_model.predict_proba(X)[:, 1]

        svm_prob = svm_model.predict_proba(X)[:, 1]

        return (dt_weight * dt_prob + svm_weight * svm_prob) / (dt_weight + svm_weight)

    masker = shap.maskers.Independent(background_df, max_samples=BACKGROUND_SIZE)

    explainer = shap.Explainer(
        probability,
        masker,
        algorithm="permutation",
        feature_names=selected_features,
        seed=42,
    )

    MAX_EVALS = 5 * (2 * len(selected_features) + 1)

    soft_shap = explainer(X_explain, max_evals=MAX_EVALS, batch_size="auto")

    np.savez_compressed(
        SOFT_SHAP_FILE,
        values=soft_shap.values,
        base_values=soft_shap.base_values,
        data=soft_shap.data,
        test_indices=explanation_indices.to_numpy(),
        feature_names=np.array(selected_features),
    )

mean_abs_shap = np.mean(np.abs(soft_shap.values), axis=0)


top_feature_index = int(np.argmax(mean_abs_shap))


top_feature = selected_features[top_feature_index]


print(f"Most influential Soft Voting feature: {top_feature}")

plt.figure()


shap.dependence_plot(
    top_feature,
    soft_shap.values,
    X_explain,
    feature_names=selected_features,
    interaction_index="auto",
    show=False,
)


plt.title(f"SHAP Dependence Plot — Soft Voting: {top_feature}")

plt.tight_layout()


DEPENDENCE_FILE = f"{SHAP_RESULTS_PATH}/soft_voting_shap_dependence.png"


plt.savefig(DEPENDENCE_FILE, dpi=300, bbox_inches="tight")

plt.show()


print(f"\nDependence plot saved to:\n{DEPENDENCE_FILE}")

## Local SHAP Explanations

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive


try:
    import shap
except ImportError:
    !pip install -q shap
    import shap


drive.mount("/content/drive")


BASE_PATH = "/content/drive/MyDrive/COMP721 Project"

SELECTION_PATH = f"{BASE_PATH}/feature_selection"

MODEL_PATH = f"{BASE_PATH}/models"

SHAP_RESULTS_PATH = f"{BASE_PATH}/phase3/shap"

os.makedirs(SHAP_RESULTS_PATH, exist_ok=True)


TRAIN_FILE = f"{SELECTION_PATH}/train_final.csv"

TEST_FILE = f"{SELECTION_PATH}/test_final.csv"

SOFT_MODEL_FILE = f"{MODEL_PATH}/soft_voting_ensemble_final.joblib"

SOFT_SHAP_FILE = f"{SHAP_RESULTS_PATH}/soft_voting_shap_values.npz"

train_df = pd.read_csv(TRAIN_FILE)

test_df = pd.read_csv(TEST_FILE)


selected_features = [col for col in train_df.columns if col != "cardio"]


X_train = train_df[selected_features]

X_test = test_df[selected_features]

y_test = test_df["cardio"]


BACKGROUND_SIZE = 50
EXPLANATION_SIZE = 200


background_df = X_train.sample(
    n=min(BACKGROUND_SIZE, len(X_train)), random_state=42
).reset_index(drop=True)


explanation_indices = X_test.sample(
    n=min(EXPLANATION_SIZE, len(X_test)), random_state=42
).index


X_explain = X_test.loc[explanation_indices].copy()

y_explain = y_test.loc[explanation_indices].copy()

artifact = joblib.load(SOFT_MODEL_FILE)


dt_model = artifact["decision_tree_model"]

svm_model = artifact["svm_pipeline"]

dt_weight = artifact["weights"]["decision_tree"]

svm_weight = artifact["weights"]["svm"]


def probability(X):

    if not isinstance(X, pd.DataFrame):
        X = pd.DataFrame(X, columns=selected_features)

    dt_prob = dt_model.predict_proba(X)[:, 1]

    svm_prob = svm_model.predict_proba(X)[:, 1]

    return (dt_weight * dt_prob + svm_weight * svm_prob) / (dt_weight + svm_weight)


if os.path.exists(SOFT_SHAP_FILE):
    saved = np.load(SOFT_SHAP_FILE, allow_pickle=True)

    soft_shap = shap.Explanation(
        values=saved["values"],
        base_values=saved["base_values"],
        data=saved["data"],
        feature_names=selected_features,
    )


else:
    print("Soft Voting SHAP values not found. Computing them...")

    masker = shap.maskers.Independent(background_df, max_samples=BACKGROUND_SIZE)

    explainer = shap.Explainer(
        probability,
        masker,
        algorithm="permutation",
        feature_names=selected_features,
        seed=42,
    )

    MAX_EVALS = 5 * (2 * len(selected_features) + 1)

    soft_shap = explainer(X_explain, max_evals=MAX_EVALS, batch_size="auto")

    np.savez_compressed(
        SOFT_SHAP_FILE,
        values=soft_shap.values,
        base_values=soft_shap.base_values,
        data=soft_shap.data,
        test_indices=explanation_indices.to_numpy(),
        feature_names=np.array(selected_features),
    )

explain_probabilities = probability(X_explain)


explain_predictions = (explain_probabilities >= 0.5).astype(int)


true_labels = y_explain.to_numpy()

local_cases = []


for target_class, case_name in [(0, "No_CVD"), (1, "CVD")]:
    candidates = np.where(
        (true_labels == target_class) & (explain_predictions == target_class)
    )[0]

    if len(candidates) == 0:
        candidates = np.where(true_labels == target_class)[0]

    if len(candidates) == 0:
        continue

    position = int(candidates[0])

    original_test_index = int(explanation_indices[position])

    probability_cvd = float(explain_probabilities[position])

    predicted_class = int(explain_predictions[position])

    true_class = int(true_labels[position])

    print("\n" + "=" * 75)

    print(f"LOCAL SHAP EXPLANATION — {case_name}")

    print("=" * 75)

    print(f"Test index:       {original_test_index}")

    print(f"True class:       {true_class}")

    print(f"Predicted class:  {predicted_class}")

    print(f"P(CVD = 1):       {probability_cvd:.4f}")

    print("\nPatient feature values:")

    display(X_explain.iloc[[position]])

    shap.plots.waterfall(
        soft_shap[position], max_display=len(selected_features), show=False
    )

    plt.title(f"Local SHAP Explanation — {case_name}")

    plt.tight_layout()

    local_plot_file = f"{SHAP_RESULTS_PATH}/local_shap_{case_name.lower()}.png"

    plt.savefig(local_plot_file, dpi=300, bbox_inches="tight")

    plt.show()

    case_record = {
        "case": case_name,
        "test_index": original_test_index,
        "true_class": true_class,
        "predicted_class": predicted_class,
        "probability_cvd": probability_cvd,
    }

    for feature in selected_features:
        case_record[feature] = float(X_explain.iloc[position][feature])

    local_cases.append(case_record)

local_cases_df = pd.DataFrame(local_cases)


LOCAL_CASES_FILE = f"{SHAP_RESULTS_PATH}/local_shap_cases.csv"


local_cases_df.to_csv(LOCAL_CASES_FILE, index=False)


print("\nLocal explanation cases:")

display(local_cases_df)


print(f"\nSaved case details to:\n{LOCAL_CASES_FILE}")

# Statistical Analysis

## Paired 5-Fold Statistical Tests

In [ ]:
import os
import json
import numpy as np
import pandas as pd

from scipy.stats import shapiro, ttest_rel, wilcoxon

from google.colab import drive

drive.mount("/content/drive")

BASE_PATH = "/content/drive/MyDrive/COMP721 Project"

DT_RESULTS_PATH = f"{BASE_PATH}/model_results/decision_tree"

SVM_RESULTS_PATH = f"{BASE_PATH}/model_results/svm"

SOFT_RESULTS_PATH = f"{BASE_PATH}/model_results/soft_voting"

STACKING_RESULTS_PATH = f"{BASE_PATH}/phase3/stacking"

ENSEMBLE_SELECTION_PATH = f"{BASE_PATH}/phase3/ensemble_selection"

STAT_RESULTS_PATH = f"{BASE_PATH}/phase3/statistical_analysis"

os.makedirs(STAT_RESULTS_PATH, exist_ok=True)


DT_CV_FILE = f"{DT_RESULTS_PATH}/decision_tree_cv_fold_scores.npy"

SVM_CV_FILE = f"{SVM_RESULTS_PATH}/svm_cv_fold_scores.npy"

SOFT_CV_FILE = f"{SOFT_RESULTS_PATH}/soft_voting_cv_fold_scores.npy"

STACKING_CV_FILE = f"{STACKING_RESULTS_PATH}/stacking_cv_fold_scores.npy"

ENSEMBLE_SELECTION_FILE = f"{ENSEMBLE_SELECTION_PATH}/ensemble_selection_summary.json"

required_files = [DT_CV_FILE, SVM_CV_FILE, ENSEMBLE_SELECTION_FILE]


for file_path in required_files:
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Required file not found:\n{file_path}")

dt_cv_scores = np.load(DT_CV_FILE).astype(float)


svm_cv_scores = np.load(SVM_CV_FILE).astype(float)

with open(ENSEMBLE_SELECTION_FILE, "r") as f:
    ensemble_selection = json.load(f)


selected_ensemble_type = ensemble_selection["selected_ensemble"]

if selected_ensemble_type == "Soft Voting":
    if not os.path.exists(SOFT_CV_FILE):
        raise FileNotFoundError(f"Soft Voting CV file not found:\n{SOFT_CV_FILE}")

    selected_ensemble_cv_scores = np.load(SOFT_CV_FILE).astype(float)


elif selected_ensemble_type == "Stacking":
    if not os.path.exists(STACKING_CV_FILE):
        raise FileNotFoundError(f"Stacking CV file not found:\n{STACKING_CV_FILE}")

    selected_ensemble_cv_scores = np.load(STACKING_CV_FILE).astype(float)


else:
    raise ValueError(f"Unexpected selected ensemble: {selected_ensemble_type}")

if len(dt_cv_scores) != 5:
    raise ValueError("Decision Tree CV scores do not contain exactly 5 folds.")


if len(svm_cv_scores) != 5:
    raise ValueError("SVM CV scores do not contain exactly 5 folds.")


if len(selected_ensemble_cv_scores) != 5:
    raise ValueError("Selected ensemble CV scores do not contain exactly 5 folds.")


if not (
    np.isfinite(dt_cv_scores).all()
    and np.isfinite(svm_cv_scores).all()
    and np.isfinite(selected_ensemble_cv_scores).all()
):
    raise ValueError("One or more CV score arrays contain invalid values.")

paired_cv_df = pd.DataFrame(
    {
        "Fold": np.arange(1, 6),
        "Decision_Tree": dt_cv_scores,
        "SVM": svm_cv_scores,
        "Selected_Ensemble": selected_ensemble_cv_scores,
    }
)


paired_cv_df["Ensemble_minus_DT"] = (
    paired_cv_df["Selected_Ensemble"] - paired_cv_df["Decision_Tree"]
)


paired_cv_df["Ensemble_minus_SVM"] = (
    paired_cv_df["Selected_Ensemble"] - paired_cv_df["SVM"]
)


print("=" * 80)

print("PAIRED 5-FOLD CROSS-VALIDATION SCORES")

print("=" * 80)

print(f"Selected ensemble: {selected_ensemble_type}")

print("\nAll comparisons use the same aligned five cross-validation folds.\n")


display(paired_cv_df.round(6))


print("\nMean CV accuracies:")

print(f"Decision Tree:     {dt_cv_scores.mean() * 100:.4f}%")

print(f"SVM:               {svm_cv_scores.mean() * 100:.4f}%")

print(
    f"{selected_ensemble_type}:"
    f"{' ' * max(1, 18 - len(selected_ensemble_type))}"
    f"{selected_ensemble_cv_scores.mean() * 100:.4f}%"
)

ALPHA = 0.05


def paired_superiority_test(
    base_model_name, base_scores, ensemble_scores, ensemble_name, alpha=0.05
):

    base_scores = np.asarray(base_scores, dtype=float)

    ensemble_scores = np.asarray(ensemble_scores, dtype=float)

    differences = ensemble_scores - base_scores

    shapiro_result = shapiro(differences)

    shapiro_statistic = float(shapiro_result.statistic)

    shapiro_p_value = float(shapiro_result.pvalue)

    normality_supported = shapiro_p_value > alpha

    if np.allclose(differences, 0.0):
        test_name = "No test required — all paired differences are zero"

        test_statistic = 0.0
        p_value = 1.0

    elif normality_supported:
        result = ttest_rel(ensemble_scores, base_scores, alternative="greater")

        test_name = "One-sided paired t-test"

        test_statistic = float(result.statistic)

        p_value = float(result.pvalue)

    else:
        result = wilcoxon(
            ensemble_scores,
            base_scores,
            alternative="greater",
            zero_method="wilcox",
            method="auto",
        )

        test_name = "One-sided Wilcoxon signed-rank test"

        test_statistic = float(result.statistic)

        p_value = float(result.pvalue)

    statistically_significant = p_value < alpha

    if statistically_significant:
        conclusion = (
            f"Reject H0. At alpha={alpha:.2f}, "
            f"there is statistically significant evidence "
            f"that {ensemble_name} achieves greater "
            f"cross-validation accuracy than "
            f"{base_model_name}."
        )

    else:
        conclusion = (
            f"Fail to reject H0. At alpha={alpha:.2f}, "
            f"there is insufficient statistical evidence "
            f"that {ensemble_name} achieves greater "
            f"cross-validation accuracy than "
            f"{base_model_name}."
        )

    record = {
        "comparison": (f"{ensemble_name} vs {base_model_name}"),
        "null_hypothesis": (
            f"{ensemble_name} does not achieve "
            f"greater paired-fold accuracy than "
            f"{base_model_name}."
        ),
        "alternative_hypothesis": (
            f"{ensemble_name} achieves greater "
            f"paired-fold accuracy than "
            f"{base_model_name}."
        ),
        "alpha": float(alpha),
        "base_mean_accuracy": float(base_scores.mean()),
        "ensemble_mean_accuracy": float(ensemble_scores.mean()),
        "mean_difference": float(differences.mean()),
        "mean_difference_percentage_points": float(differences.mean() * 100),
        "paired_differences": differences.tolist(),
        "shapiro_statistic": shapiro_statistic,
        "shapiro_p_value": shapiro_p_value,
        "normality_supported": bool(normality_supported),
        "test_used": test_name,
        "test_statistic": test_statistic,
        "p_value": p_value,
        "statistically_significant": bool(statistically_significant),
        "conclusion": conclusion,
    }

    print("\n" + "=" * 80)

    print(f"{ensemble_name.upper()} VS {base_model_name.upper()}")

    print("=" * 80)

    print("\nH0:")

    print(
        f"{ensemble_name} does not achieve greater "
        f"paired-fold accuracy than {base_model_name}."
    )

    print("\nH1:")

    print(
        f"{ensemble_name} achieves greater paired-fold accuracy than {base_model_name}."
    )

    print(f"\nSignificance level: alpha = {alpha}")

    print("\nPaired differences (ensemble - base):")

    print(np.round(differences, 8))

    print(f"\n{base_model_name} mean accuracy: {base_scores.mean() * 100:.4f}%")

    print(f"{ensemble_name} mean accuracy: {ensemble_scores.mean() * 100:.4f}%")

    print(f"Mean improvement: {differences.mean() * 100:.4f} percentage points")

    print("\nShapiro-Wilk normality test:")

    print(f"  Statistic = {shapiro_statistic:.6f}")

    print(f"  p-value   = {shapiro_p_value:.6f}")

    print(f"  Normality supported = {normality_supported}")

    print(f"\nInferential test: {test_name}")

    print(f"  Statistic = {test_statistic:.6f}")

    print(f"  p-value   = {p_value:.6f}")

    print(f"\nConclusion:\n{conclusion}")

    return record


dt_vs_ensemble = paired_superiority_test(
    base_model_name="Decision Tree",
    base_scores=dt_cv_scores,
    ensemble_scores=selected_ensemble_cv_scores,
    ensemble_name=selected_ensemble_type,
    alpha=ALPHA,
)

svm_vs_ensemble = paired_superiority_test(
    base_model_name="SVM",
    base_scores=svm_cv_scores,
    ensemble_scores=selected_ensemble_cv_scores,
    ensemble_name=selected_ensemble_type,
    alpha=ALPHA,
)

statistical_results = [dt_vs_ensemble, svm_vs_ensemble]


statistical_results_df = pd.DataFrame(statistical_results)


summary_columns = [
    "comparison",
    "base_mean_accuracy",
    "ensemble_mean_accuracy",
    "mean_difference_percentage_points",
    "shapiro_statistic",
    "shapiro_p_value",
    "normality_supported",
    "test_used",
    "test_statistic",
    "p_value",
    "statistically_significant",
    "conclusion",
]


print("\n" + "=" * 80)

print("FINAL STATISTICAL ANALYSIS SUMMARY")

print("=" * 80)


display(statistical_results_df[summary_columns].round(6))

PAIRED_CV_FILE = f"{STAT_RESULTS_PATH}/paired_cv_scores.csv"

STATISTICAL_RESULTS_CSV = f"{STAT_RESULTS_PATH}/statistical_test_results.csv"

STATISTICAL_RESULTS_JSON = f"{STAT_RESULTS_PATH}/statistical_test_results.json"


paired_cv_df.to_csv(PAIRED_CV_FILE, index=False)


statistical_results_df.to_csv(STATISTICAL_RESULTS_CSV, index=False)


with open(STATISTICAL_RESULTS_JSON, "w") as f:
    json.dump(statistical_results, f, indent=4)


print("\nSaved statistical-analysis artifacts:")

print(f"\nPaired CV scores:\n{PAIRED_CV_FILE}")

print(f"\nStatistical results CSV:\n{STATISTICAL_RESULTS_CSV}")

print(f"\nStatistical results JSON:\n{STATISTICAL_RESULTS_JSON}")


print("\nStatistical analysis complete.")